In [1]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

Cloning into 'LHL-final-final-project'...
remote: Enumerating objects: 410, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 410 (delta 95), reused 50 (delta 12), pack-reused 224 (from 1)
Receiving objects: 100% (410/410), 6.11 MiB | 4.59 MiB/s, done.
Resolving deltas: 100% (210/210), done.


In [2]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup, Comment
import requests
from io import StringIO
import string
import time
import re
import os

In [3]:
# load the player game logs CSV from the data folder
df_gamelogs = pd.read_csv("LHL-final-final-project/data/2020-2023_basketball_reference_gamelog.csv")


# preview
df_gamelogs.head()

,team,year,home_away,opp,win_loss,team_score,opp_score,team_fg,team_fga,team_fg_pct,...,opponent_ft_pct,opponent_orb,opponent_trb,opponent_ast,opponent_stl,opponent_blk,opponent_tov,opponent_pf,day,month
0,ATL,2020,NaN,DAL,W,105.0,95.0,34,62,.548,...,.700,8,29,18,10,1,13,27,26.0,7.0
1,ATL,2020,@,LVA,L,70.0,100.0,28,70,.400,...,.692,14,47,16,11,1,19,14,29.0,7.0
2,ATL,2020,NaN,NYL,W,84.0,78.0,28,75,.373,...,.882,8,33,13,8,11,15,23,31.0,7.0
3,ATL,2020,@,IND,L,77.0,93.0,33,69,.478,...,.810,9,32,26,5,5,15,13,2.0,8.0
4,ATL,2020,NaN,PHO,L,74.0,81.0,29,62,.468,...,.864,10,33,19,7,2,10,18,4.0,8.0


In [4]:
# load the player game logs CSV from the data folder
df_2024_gamelogs = pd.read_csv("LHL-final-final-project/data/2024_merged_gamelogs.csv")


# preview
df_2024_gamelogs.head()

,team,g_num,month,day,home_away,opp,win_loss,team_score,opp_score,team_fg,...,day_of_week_by_team_travel_distance,team_vs_opp_median_score_by_team_travel_distance,team_vs_opp_homeaway_median_score_by_team_travel_distance,team_home_or_away_median_score_by_team_travel_distance,team_home_or_away_median_allowed_by_team_travel_distance,team_day_median_score_by_team_travel_distance,team_day_median_allowed_by_team_travel_distance,travel_distance_by_team_travel_distance,median_score_for_by_team_travel_distance,median_score_against_by_team_travel_distance
0,ATL,1,5,15,2,LAS,1,92,81,34,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
1,ATL,2,5,18,2,PHO,2,85,88,27,...,2.0,76.0,76.0,78.0,80.5,76.0,80.0,1.0,77.5,77.5
2,ATL,3,5,21,1,DAL,1,83,78,30,...,4.0,81.0,75.5,78.0,80.5,73.0,78.0,3.0,81.0,85.0
3,ATL,4,5,26,1,MIN,2,79,92,31,...,3.0,75.0,70.0,78.5,79.5,76.0,80.0,0.0,77.0,79.5
4,ATL,5,5,29,2,WAS,1,73,67,26,...,4.0,75.0,76.5,78.0,80.5,76.0,80.0,2.0,78.0,80.0


In [6]:
team = 'ATL'
year = 2023
url = f"https://www.basketball-reference.com/wnba/teams/{team}/{year}/gamelog-advanced/"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.content, "html.parser")
table = soup.find("table", id="wnba_tgl_advanced")
print(table is not None)  # Should print True if table exists

True


In [7]:
# Team codes remain the same across years
teams = ['ATL', 'CHI', 'CON', 'DAL', 'IND', 'LAS', 'MIN', 'NYL', 'PHO', 'SEA', 'WAS', 'LVA']

# Base URL with placeholder for team and year
base_url = "https://www.basketball-reference.com/wnba/teams/{team}/{year}/gamelog-advanced/"

# Empty list to store DataFrames
frames = []

# Loop through each year and team
for year in range(2020, 2024):
    for team in teams:
        url = base_url.format(team=team, year=year)
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers)
        time.sleep(5)  # Avoid hitting rate limits

        soup = BeautifulSoup(response.content, "html.parser")
        table = soup.find("table", id="wnba_tgl_advanced")

        if table is not None:
            df_team = pd.read_html(StringIO(str(table)))[0]

            # Combine multi-level headers
            df_team.columns = [f"{a}_{b}" for a, b in df_team.columns]

            # Drop extra unnamed columns
            drop_cols = [
                'Unnamed: 6_level_0_Unnamed: 6_level_1',
                'Unnamed: 9_level_0_Unnamed: 9_level_1',
                'Unnamed: 26_level_0_Unnamed: 26_level_1'
            ]
            df_team = df_team.drop(columns=drop_cols, errors='ignore')

            # Rename key columns
            df_team = df_team.rename(columns={
                'Unnamed: 0_level_0_Rk': 'Rk',
                'Unnamed: 1_level_0_G#': 'G#',
                'Unnamed: 2_level_0_Date': 'Date',
                'Unnamed: 4_level_0_Opp': 'Opp',
                'Unnamed: 5_level_0_W/L': 'W/L',
                'Unnamed: 7_level_0_Tm': 'Team_Score',
                'Unnamed: 8_level_0_Opp': 'Opp_Score'
            })

            # Add team and year columns at the beginning
            df_team.insert(0, "Year", year)
            df_team.insert(0, "Team", team)

            frames.append(df_team)
        else:
            print(f"No table found for {team}, {year}")

# Combine all collected DataFrames into team_gamelog_df
team_advanced_gamelog_df = pd.concat(frames, ignore_index=True)

In [8]:
team_advanced_gamelog_df.shape

(1656, 30)

In [9]:
team_advanced_gamelog_df.head()

,Team,Year,Rk,Unnamed: 1_level_0_Date,Unnamed: 2_level_0_Unnamed: 2_level_1,Unnamed: 3_level_0_Opp,Unnamed: 4_level_0_W/L,Unnamed: 5_level_0_Tm,Unnamed: 6_level_0_Opp,Unnamed: 7_level_0_Unnamed: 7_level_1,...,Unnamed: 18_level_0_Unnamed: 18_level_1,Offensive Four Factors_eFG%,Offensive Four Factors_TOV%,Offensive Four Factors_ORB%,Offensive Four Factors_FT/FGA,Unnamed: 23_level_0_Unnamed: 23_level_1,Defensive Four Factors_eFG%,Defensive Four Factors_TOV%,Defensive Four Factors_DRB%,Defensive Four Factors_FT/FGA
0,ATL,2020,1,2020-07-26,NaN,DAL,W,105,95,NaN,...,NaN,.605,17.2,16.0,.484,NaN,.519,13.0,78.9,.179
1,ATL,2020,2,2020-07-29,@,LVA,L,70,100,NaN,...,NaN,.436,17.4,17.5,.129,NaN,.577,18.7,56.3,.254
2,ATL,2020,3,2020-07-31,NaN,NYL,W,84,78,NaN,...,NaN,.393,14.5,40.5,.333,NaN,.438,15.9,76.5,.208
3,ATL,2020,4,2020-08-02,@,IND,L,77,93,NaN,...,NaN,.507,14.2,17.9,.101,NaN,.576,16.6,70.0,.258
4,ATL,2020,5,2020-08-04,NaN,PHO,L,74,81,NaN,...,NaN,.500,25.4,17.9,.194,NaN,.413,10.6,75.0,.253


In [11]:
# Flatten multi-level column headers
team_advanced_gamelog_df.columns = [col.split('_')[-1] if 'Unnamed' in col else col for col in team_advanced_gamelog_df.columns]

# Verify flattened columns
print(team_advanced_gamelog_df.columns.tolist())

['Team', 'Year', 'Rk', 'Date', '1', 'Opp', 'W/L', 'Tm', 'Opp', '1', 'Advanced_ORtg', 'Advanced_DRtg', 'Advanced_Pace', 'Advanced_FTr', 'Advanced_3PAr', 'Advanced_TS%', 'Advanced_TRB%', 'Advanced_AST%', 'Advanced_STL%', 'Advanced_BLK%', '1', 'Offensive Four Factors_eFG%', 'Offensive Four Factors_TOV%', 'Offensive Four Factors_ORB%', 'Offensive Four Factors_FT/FGA', '1', 'Defensive Four Factors_eFG%', 'Defensive Four Factors_TOV%', 'Defensive Four Factors_DRB%', 'Defensive Four Factors_FT/FGA']


In [12]:
for col in team_advanced_gamelog_df.columns:
    print(col)


Team
Year
Rk
Date
1
Opp
W/L
Tm
Opp
1
Advanced_ORtg
Advanced_DRtg
Advanced_Pace
Advanced_FTr
Advanced_3PAr
Advanced_TS%
Advanced_TRB%
Advanced_AST%
Advanced_STL%
Advanced_BLK%
1
Offensive Four Factors_eFG%
Offensive Four Factors_TOV%
Offensive Four Factors_ORB%
Offensive Four Factors_FT/FGA
1
Defensive Four Factors_eFG%
Defensive Four Factors_TOV%
Defensive Four Factors_DRB%
Defensive Four Factors_FT/FGA


In [14]:
correct_headers = [
    'Team', 'Year', 'Rk', 'Date', 'home_away', 'Opp', 'win_loss', 'team_score', 'opp_score', 'delete1',
    'Advanced_ORtg', 'Advanced_DRtg', 'Advanced_Pace', 'Advanced_FTr', 'Advanced_3PAr',
    'Advanced_TS%', 'Advanced_TRB%', 'Advanced_AST%', 'Advanced_STL%', 'Advanced_BLK%',
    'delete2', 'Offensive_Four_Factors_eFG%', 'Offensive_Four_Factors_TOV%',
    'Offensive_Four_Factors_ORB%', 'Offensive_Four_Factors_FT/FGA', 'delete3',
    'Defensive_Four_Factors_eFG%', 'Defensive_Four_Factors_TOV%',
    'Defensive_Four_Factors_DRB%', 'Defensive_Four_Factors_FT/FGA'
]

In [15]:
team_advanced_gamelog_df.columns = correct_headers

# Quick check
print(team_advanced_gamelog_df.columns.tolist())

['Team', 'Year', 'Rk', 'Date', 'home_away', 'Opp', 'win_loss', 'team_score', 'opp_score', 'delete1', 'Advanced_ORtg', 'Advanced_DRtg', 'Advanced_Pace', 'Advanced_FTr', 'Advanced_3PAr', 'Advanced_TS%', 'Advanced_TRB%', 'Advanced_AST%', 'Advanced_STL%', 'Advanced_BLK%', 'delete2', 'Offensive_Four_Factors_eFG%', 'Offensive_Four_Factors_TOV%', 'Offensive_Four_Factors_ORB%', 'Offensive_Four_Factors_FT/FGA', 'delete3', 'Defensive_Four_Factors_eFG%', 'Defensive_Four_Factors_TOV%', 'Defensive_Four_Factors_DRB%', 'Defensive_Four_Factors_FT/FGA']


In [16]:
for col in team_advanced_gamelog_df.columns:
    print(col)

Team
Year
Rk
Date
home_away
Opp
win_loss
team_score
opp_score
delete1
Advanced_ORtg
Advanced_DRtg
Advanced_Pace
Advanced_FTr
Advanced_3PAr
Advanced_TS%
Advanced_TRB%
Advanced_AST%
Advanced_STL%
Advanced_BLK%
delete2
Offensive_Four_Factors_eFG%
Offensive_Four_Factors_TOV%
Offensive_Four_Factors_ORB%
Offensive_Four_Factors_FT/FGA
delete3
Defensive_Four_Factors_eFG%
Defensive_Four_Factors_TOV%
Defensive_Four_Factors_DRB%
Defensive_Four_Factors_FT/FGA


In [17]:
# Drop unnecessary columns
team_advanced_gamelog_df = team_advanced_gamelog_df.drop(columns=['delete1', 'delete2', 'delete3'])

# Verify columns after drop
print(team_advanced_gamelog_df.columns.tolist())

['Team', 'Year', 'Rk', 'Date', 'home_away', 'Opp', 'win_loss', 'team_score', 'opp_score', 'Advanced_ORtg', 'Advanced_DRtg', 'Advanced_Pace', 'Advanced_FTr', 'Advanced_3PAr', 'Advanced_TS%', 'Advanced_TRB%', 'Advanced_AST%', 'Advanced_STL%', 'Advanced_BLK%', 'Offensive_Four_Factors_eFG%', 'Offensive_Four_Factors_TOV%', 'Offensive_Four_Factors_ORB%', 'Offensive_Four_Factors_FT/FGA', 'Defensive_Four_Factors_eFG%', 'Defensive_Four_Factors_TOV%', 'Defensive_Four_Factors_DRB%', 'Defensive_Four_Factors_FT/FGA']


In [18]:
# Lowercase all column names
team_advanced_gamelog_df.columns = team_advanced_gamelog_df.columns.str.lower()

In [20]:
# Drop rows where 'opp' column is null
team_advanced_gamelog_df = team_advanced_gamelog_df.dropna(subset=['opp'])

# Verify rows dropped
print(team_advanced_gamelog_df['opp'].isnull().sum())  # should print 0

0


In [22]:
# Remove header rows mistakenly included in data
team_advanced_gamelog_df = team_advanced_gamelog_df[team_advanced_gamelog_df['date'] != 'Date']

# Confirm the rows were removed
print((team_advanced_gamelog_df['date'] == 'Date').sum())  # should print 0

0


In [23]:
# Ensure 'date' is datetime
team_advanced_gamelog_df['date'] = pd.to_datetime(team_advanced_gamelog_df['date'])

# Create 'day' and 'month' columns
team_advanced_gamelog_df['day'] = team_advanced_gamelog_df['date'].dt.day
team_advanced_gamelog_df['month'] = team_advanced_gamelog_df['date'].dt.month

# Drop original 'date' column
team_advanced_gamelog_df = team_advanced_gamelog_df.drop(columns=['date'])

In [24]:
# Replace specific characters in column names
team_advanced_gamelog_df.columns = (
    team_advanced_gamelog_df.columns
    .str.replace('%', '_pct', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

In [26]:
# Drop original 'rk' column
team_advanced_gamelog_df = team_advanced_gamelog_df.drop(columns=['rk'])

In [27]:
for col in team_advanced_gamelog_df.columns:
    print(col)


team
year
home_away
opp
win_loss
team_score
opp_score
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
offensive_four_factors_ft_fga
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
defensive_four_factors_ft_fga
day
month


In [28]:
df_gamelogs.shape

(1608, 41)

In [29]:
team_advanced_gamelog_df.shape

(1560, 27)

In [31]:
print(df_gamelogs['team_score'].dtype)
print(team_advanced_gamelog_df['team_score'].dtype)

float64
object


In [32]:
# Convert 'team_score' to numeric safely
team_advanced_gamelog_df['team_score'] = pd.to_numeric(team_advanced_gamelog_df['team_score'], errors='coerce')

In [34]:
# Convert 'team_score' to numeric safely
team_advanced_gamelog_df['opp_score'] = pd.to_numeric(team_advanced_gamelog_df['opp_score'], errors='coerce')

In [35]:
# Merge DataFrames on specified columns
df_merged_gamelogs = pd.merge(
    df_gamelogs,
    team_advanced_gamelog_df,
    on=['team', 'year', 'home_away', 'opp', 'win_loss', 'day', 'month', 'team_score', 'opp_score'],
    how='left'
)

# Verify merged result
print(df_merged_gamelogs.head())
print(df_merged_gamelogs.shape)

  team  year home_away  opp win_loss  team_score  opp_score team_fg team_fga  \
0  ATL  2020       NaN  DAL        W       105.0       95.0      34       62   
1  ATL  2020         @  LVA        L        70.0      100.0      28       70   
2  ATL  2020       NaN  NYL        W        84.0       78.0      28       75   
3  ATL  2020         @  IND        L        77.0       93.0      33       69   
4  ATL  2020       NaN  PHO        L        74.0       81.0      29       62   

  team_fg_pct  ... advanced_stl_pct advanced_blk_pct  \
0        .548  ...              8.0              8.0   
1        .400  ...             14.2              8.1   
2        .373  ...              9.7              4.7   
3        .478  ...              7.7              0.0   
4        .468  ...              8.5              3.6   

  offensive_four_factors_efg_pct offensive_four_factors_tov_pct  \
0                           .605                           17.2   
1                           .436                

In [36]:
# Check columns with nulls, sorted descending
print(df_merged_gamelogs.isnull().sum()[lambda x: x > 0].sort_values(ascending=False))

home_away                         828
opp                                48
win_loss                           48
team_score                         48
opp_score                          48
month                              48
day                                48
advanced_ortg                      48
advanced_drtg                      48
advanced_ftr                       48
advanced_pace                      48
offensive_four_factors_efg_pct     48
offensive_four_factors_tov_pct     48
advanced_3par                      48
advanced_ts_pct                    48
advanced_trb_pct                   48
advanced_ast_pct                   48
advanced_stl_pct                   48
advanced_blk_pct                   48
defensive_four_factors_efg_pct     48
defensive_four_factors_tov_pct     48
offensive_four_factors_orb_pct     48
offensive_four_factors_ft_fga      48
defensive_four_factors_drb_pct     48
defensive_four_factors_ft_fga      48
team_ft_pct                         1
opponent_ft_

In [37]:
# Convert 'home_away': '@' -> 0, else -> 1
df_merged_gamelogs['home_away'] = df_merged_gamelogs['home_away'].apply(lambda x: 0 if x == '@' else 1)

In [39]:
# Drop rows with null 'opp'
df_merged_gamelogs = df_merged_gamelogs.dropna(subset=['opp'])

# Quick verification
print(df_merged_gamelogs['opp'].isnull().sum())  # Should print 0

0


In [40]:
# Check columns with nulls, sorted descending
print(df_merged_gamelogs.isnull().sum()[lambda x: x > 0].sort_values(ascending=False))

team_ft_pct        1
opponent_ft_pct    1
dtype: int64


In [41]:
# Replace nulls with 0 in specified columns
df_merged_gamelogs[['team_ft_pct', 'opponent_ft_pct']] = df_merged_gamelogs[['team_ft_pct', 'opponent_ft_pct']].fillna(0)

# Quick verification
print(df_merged_gamelogs[['team_ft_pct', 'opponent_ft_pct']].isnull().sum())

team_ft_pct        0
opponent_ft_pct    0
dtype: int64


In [42]:
for col in df_merged_gamelogs.columns:
    print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
offensive_four_factors_ft_fga
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
defensive_four_factors_ft_fga


In [44]:
# Add 'year' column set to 2024
df_2024_gamelogs['year'] = 2024

In [45]:
# Columns unique to df_merged_gamelogs
unique_to_merged = set(df_merged_gamelogs.columns) - set(df_2024_gamelogs.columns)

# Columns unique to df_2024_gamelogs
unique_to_2024 = set(df_2024_gamelogs.columns) - set(df_merged_gamelogs.columns)

# Print clearly
print("Unique to df_merged_gamelogs:", unique_to_merged)
print("Unique to df_2024_gamelogs:", unique_to_2024)

Unique to df_merged_gamelogs: {'defensive_four_factors_ft_fga', 'offensive_four_factors_ft_fga'}
Unique to df_2024_gamelogs: {'defensive_four_factors_efg_pct_by_team_home_away', 'advanced_3par_by_team_day_of_week', 'team_fta_by_team_day_of_week', 'opponent_tov_by_team_travel_distance', 'team_ft_by_team_day_of_week', 'offensive_four_factors_ft_per_fga_by_team_day_of_week', 'team_day_median_allowed_by_team_travel_distance', 'team_pf_by_team_home_away', 'advanced_ftr_by_team_home_away', 'team_day_median_score_by_team_day_of_week', 'defensive_four_factors_tov_pct_by_team_travel_distance', 'team_ft_pct_by_team_home_away', 'travel_distance_by_team_day_of_week', 'team_stl_by_team_home_away', 'advanced_trb_pct_by_team_day_of_week', 'advanced_stl_pct_by_team_day_of_week', 'advanced_ortg_by_team_day_of_week', 'team_vs_opp_homeaway_median_score_by_team_day_of_week', 'team_home_or_away_median_score', 'advanced_ftr_by_team_day_of_week', 'team_orb_by_team_day_of_week', 'team_home_or_away_median_scor

In [46]:
# Concatenate vertically into df
df = pd.concat([df_merged_gamelogs, df_2024_gamelogs], ignore_index=True)

# Verify shape clearly
print(df.shape)
print(df.head())

(2040, 264)
  team  year  home_away  opp win_loss  team_score  opp_score team_fg team_fga  \
0  ATL  2020          1  DAL        W       105.0       95.0      34       62   
1  ATL  2020          0  LVA        L        70.0      100.0      28       70   
2  ATL  2020          1  NYL        W        84.0       78.0      28       75   
3  ATL  2020          0  IND        L        77.0       93.0      33       69   
4  ATL  2020          1  PHO        L        74.0       81.0      29       62   

  team_fg_pct  ... day_of_week_by_team_travel_distance  \
0        .548  ...                                 NaN   
1        .400  ...                                 NaN   
2        .373  ...                                 NaN   
3        .478  ...                                 NaN   
4        .468  ...                                 NaN   

  team_vs_opp_median_score_by_team_travel_distance  \
0                                              NaN   
1                                           

In [48]:
# Check columns with nulls, sorted descending
print(df.isnull().sum()[lambda x: x > 0].sort_values(ascending=False))

g_num                                            1560
defensive_four_factors_ft_per_fga                1560
offensive_four_factors_ft_per_fga                1560
day_of_week                                      1560
team_vs_opp_median_score                         1560
                                                 ... 
team_day_median_score_by_team_travel_distance    1560
median_score_for_by_team_travel_distance         1560
median_score_against_by_team_travel_distance     1560
offensive_four_factors_ft_fga                     480
defensive_four_factors_ft_fga                     480
Length: 207, dtype: int64


In [49]:
# Drop columns with any null values
df = df.dropna(axis=1)

# Verify no null columns remain
print(df.isnull().sum().sum())  # Should print 0
print(df.shape)  # Confirm new dimensions

0
(2040, 57)


In [50]:
# load the player game logs CSV from the data folder
df_personas = pd.read_csv("LHL-final-final-project/data/all_persona_and_team_data.csv")


# preview
df_personas.head()

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,team_3par,team_ts%,team_efg%,team_tov%,team_orb%,team_ft/fga,team_efg%.1,team_tov%.1,team_drb%,team_ft/fga.1
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231


In [51]:
for col in df_personas.columns:
    print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
team_per_game_fg
team_per_game_fga
team_per_game_fg%
team_per_game_3p
team_per_game_3pa
team_per_game_3p%
team_per_game_2p
team_per_game_2pa
team_per_game_2p%
team_per_game_ft
team_per_game_fta
team_per_game_ft%
team_per_game_orb
team_per_game_drb
team_per_game_trb
team_per_game_ast
team_per_game_stl
team_per_game_blk
team_per_game_tov
team_per_game_pf
team_per_game_pts
opp_per_game_fg
opp_per_game_fga
opp_per_game_fg%
opp_per_game_3p
opp_per_game_3pa
opp_per_game_3p%
opp_per_game_2p
opp_per_game_2pa
opp_per_game_2p%
op

In [52]:
# List of columns to merge
persona_cols = [
    'team', 'year', 'opp', 'day', 'month',
    'all_around_star', 'and_one_machine', 'catch_and_shoot', 'corner_3_specialist',
    'defensive_anchor', 'defensive_rebounder', 'efficient_scorer', 'elite_scorer',
    'fast_break_threat', 'floor_general', 'free_throw_generator', 'glass_cleaner',
    'heave_chucker', 'impact_bench', 'midrange_sniper', 'offensive_hub',
    'offensive_rebounder', 'playmaker', 'plus_minus_driver', 'rim_protector',
    'self_creator', 'slasher', 'steal_artist', 'stretch_big',
    'three_point_specialist', 'turnover_prone', 'volume_shooter'
]

# Merge clearly into df
df = pd.merge(
    df,
    df_personas[persona_cols],
    on=['team', 'year', 'opp', 'day', 'month'],
    how='left'
)

# Verify merge results
print(df.shape)
print(df.head())

(2040, 84)
  team  year  home_away  opp win_loss  team_score  opp_score team_fg team_fga  \
0  ATL  2020          1  DAL        W       105.0       95.0      34       62   
1  ATL  2020          0  LVA        L        70.0      100.0      28       70   
2  ATL  2020          1  NYL        W        84.0       78.0      28       75   
3  ATL  2020          0  IND        L        77.0       93.0      33       69   
4  ATL  2020          1  PHO        L        74.0       81.0      29       62   

  team_fg_pct  ... playmaker plus_minus_driver rim_protector self_creator  \
0        .548  ...       0.0               0.0           0.0          1.0   
1        .400  ...       0.0               0.0           0.0          1.0   
2        .373  ...       0.0               0.0           0.0          1.0   
3        .478  ...       0.0               0.0           0.0          1.0   
4        .468  ...       0.0               0.0           0.0          1.0   

  slasher steal_artist stretch_big thre

In [53]:
for col in df.columns:
  print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midran

In [55]:
# Group by 'team' and sort clearly
df = df.sort_values(by=['team', 'year', 'month', 'day'], ascending=[True, True, True, True])

# Verify sorting
print(df[['team', 'year', 'month', 'day']].head(90))

    team  year  month   day
0    ATL  2020    7.0  26.0
1    ATL  2020    7.0  29.0
2    ATL  2020    7.0  31.0
3    ATL  2020    8.0   2.0
4    ATL  2020    8.0   4.0
..   ...   ...    ...   ...
679  ATL  2022    8.0   5.0
680  ATL  2022    8.0   7.0
681  ATL  2022    8.0   9.0
682  ATL  2022    8.0  12.0
683  ATL  2022    8.0  14.0

[90 rows x 4 columns]


In [57]:
# Create 'game_num' column counting each team's games
df['game_num'] = df.groupby('team').cumcount() + 1

# Verify results
print(df[['team', 'game_num']])

     team  game_num
0     ATL         1
1     ATL         2
2     ATL         3
3     ATL         4
4     ATL         5
...   ...       ...
2035  WAS       166
2036  WAS       167
2037  WAS       168
2038  WAS       169
2039  WAS       170

[2040 rows x 2 columns]


In [59]:
# Compute YTD mean team_score (excluding current game)
df['ytd_team_score_mean'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().expanding().mean())
      .fillna(0)
)

# Verify clearly
print(df[['team', 'year', 'team_score', 'ytd_team_score_mean']])

     team  year  team_score  ytd_team_score_mean
0     ATL  2020       105.0             0.000000
1     ATL  2020        70.0           105.000000
2     ATL  2020        84.0            87.500000
3     ATL  2020        77.0            86.333333
4     ATL  2020        74.0            84.000000
...   ...   ...         ...                  ...
2035  WAS  2024        89.0            79.285714
2036  WAS  2024        72.0            79.555556
2037  WAS  2024        73.0            79.351351
2038  WAS  2024        71.0            79.184211
2039  WAS  2024        92.0            78.974359

[2040 rows x 4 columns]


In [60]:
# Compute YTD median team_score (excluding current game)
df['ytd_team_score_median'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().expanding().median())
      .fillna(0)
)

# Verify result clearly
print(df[['team', 'year', 'team_score', 'ytd_team_score_median']].head(15))

   team  year  team_score  ytd_team_score_median
0   ATL  2020       105.0                    0.0
1   ATL  2020        70.0                  105.0
2   ATL  2020        84.0                   87.5
3   ATL  2020        77.0                   84.0
4   ATL  2020        74.0                   80.5
5   ATL  2020        92.0                   77.0
6   ATL  2020        75.0                   80.5
7   ATL  2020        82.0                   77.0
8   ATL  2020        63.0                   79.5
9   ATL  2020        80.0                   77.0
10  ATL  2020        67.0                   78.5
11  ATL  2020        91.0                   77.0
12  ATL  2020        85.0                   78.5
13  ATL  2020        78.0                   80.0
14  ATL  2020        79.0                   79.0


In [61]:
# Compute YTD minimum team_score (excluding current game)
df['ytd_team_score_min'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().expanding().min())
      .fillna(0)
)

# Verify result clearly
print(df[['team', 'year', 'team_score', 'ytd_team_score_min']].head(15))

   team  year  team_score  ytd_team_score_min
0   ATL  2020       105.0                 0.0
1   ATL  2020        70.0               105.0
2   ATL  2020        84.0                70.0
3   ATL  2020        77.0                70.0
4   ATL  2020        74.0                70.0
5   ATL  2020        92.0                70.0
6   ATL  2020        75.0                70.0
7   ATL  2020        82.0                70.0
8   ATL  2020        63.0                70.0
9   ATL  2020        80.0                63.0
10  ATL  2020        67.0                63.0
11  ATL  2020        91.0                63.0
12  ATL  2020        85.0                63.0
13  ATL  2020        78.0                63.0
14  ATL  2020        79.0                63.0


In [62]:
# Compute YTD maximum team_score (excluding current game)
df['ytd_team_score_max'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().expanding().max())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'team_score', 'ytd_team_score_max']].head(15))

   team  year  team_score  ytd_team_score_max
0   ATL  2020       105.0                 0.0
1   ATL  2020        70.0               105.0
2   ATL  2020        84.0               105.0
3   ATL  2020        77.0               105.0
4   ATL  2020        74.0               105.0
5   ATL  2020        92.0               105.0
6   ATL  2020        75.0               105.0
7   ATL  2020        82.0               105.0
8   ATL  2020        63.0               105.0
9   ATL  2020        80.0               105.0
10  ATL  2020        67.0               105.0
11  ATL  2020        91.0               105.0
12  ATL  2020        85.0               105.0
13  ATL  2020        78.0               105.0
14  ATL  2020        79.0               105.0


In [76]:
def correct_ytd_opp_mean(row):
    opp_games = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]
    return opp_games['team_score'].mean() if not opp_games.empty else 0

# Apply directly and safely into df
df['ytd_opp_score_mean'] = df.apply(correct_ytd_opp_mean, axis=1)

# Verify clearly with your requested print
print(
    df.loc[
        (df['year'] == 2020) & ((df['team'] == 'NYL') | (df['opp'] == 'NYL')),
        ['team', 'opp', 'year', 'month', 'day', 'team_score', 'ytd_team_score_mean', 'ytd_opp_score_mean']
    ].sort_values(['month', 'day']).to_string(index=False)
)

team opp  year  month  day  team_score  ytd_team_score_mean  ytd_opp_score_mean
 NYL SEA  2020    7.0 25.0        71.0             0.000000            0.000000
 SEA NYL  2020    7.0 25.0        87.0             0.000000            0.000000
 DAL NYL  2020    7.0 29.0        93.0            95.000000           71.000000
 NYL DAL  2020    7.0 29.0        80.0            71.000000           95.000000
 ATL NYL  2020    7.0 31.0        84.0            87.500000           75.500000
 NYL ATL  2020    7.0 31.0        78.0            75.500000           87.500000
 NYL PHO  2020    8.0  2.0        67.0            76.333333           92.666667
 PHO NYL  2020    8.0  2.0        96.0            92.666667           76.333333
 MIN NYL  2020    8.0  5.0        92.0            76.000000           74.000000
 NYL MIN  2020    8.0  5.0        66.0            74.000000           76.000000
 NYL WAS  2020    8.0  7.0        74.0            72.400000           89.400000
 WAS NYL  2020    8.0  7.0        66.0  

In [81]:
def correct_ytd_opp_median(row):
    opp_games = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]
    return opp_games['team_score'].median() if not opp_games.empty else 0

# Apply clearly and safely
df['ytd_opp_score_median'] = df.apply(correct_ytd_opp_median, axis=1)

# Quick verification (same NYL example)
print(
    df.loc[
        (df['year'] == 2020) & ((df['team'] == 'NYL') | (df['opp'] == 'NYL')),
        ['team', 'opp', 'year', 'month', 'day', 'team_score', 'ytd_opp_score_median']
    ].sort_values(['month', 'day']).to_string(index=False)
)

team opp  year  month  day  team_score  ytd_opp_score_median
 NYL SEA  2020    7.0 25.0        71.0                   0.0
 SEA NYL  2020    7.0 25.0        87.0                   0.0
 DAL NYL  2020    7.0 29.0        93.0                  71.0
 NYL DAL  2020    7.0 29.0        80.0                  95.0
 ATL NYL  2020    7.0 31.0        84.0                  75.5
 NYL ATL  2020    7.0 31.0        78.0                  87.5
 NYL PHO  2020    8.0  2.0        67.0                 100.0
 PHO NYL  2020    8.0  2.0        96.0                  78.0
 MIN NYL  2020    8.0  5.0        92.0                  74.5
 NYL MIN  2020    8.0  5.0        66.0                  77.5
 NYL WAS  2020    8.0  7.0        74.0                  89.0
 WAS NYL  2020    8.0  7.0        66.0                  71.0
 LVA NYL  2020    8.0  9.0        78.0                  72.5
 NYL LVA  2020    8.0  9.0        76.0                  86.0
 LAS NYL  2020    8.0 11.0        93.0                  74.0
 NYL LAS  2020    8.0 11

In [82]:
def correct_ytd_opp_min(row):
    opp_games = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]
    return opp_games['team_score'].min() if not opp_games.empty else 0

# Apply clearly
df['ytd_opp_score_min'] = df.apply(correct_ytd_opp_min, axis=1)

# Quick verification clearly
print(
    df.loc[
        (df['year'] == 2020) & ((df['team'] == 'NYL') | (df['opp'] == 'NYL')),
        ['team', 'opp', 'year', 'month', 'day', 'team_score', 'ytd_opp_score_min']
    ].sort_values(['month', 'day']).to_string(index=False)
)

team opp  year  month  day  team_score  ytd_opp_score_min
 NYL SEA  2020    7.0 25.0        71.0                0.0
 SEA NYL  2020    7.0 25.0        87.0                0.0
 DAL NYL  2020    7.0 29.0        93.0               71.0
 NYL DAL  2020    7.0 29.0        80.0               95.0
 ATL NYL  2020    7.0 31.0        84.0               71.0
 NYL ATL  2020    7.0 31.0        78.0               70.0
 NYL PHO  2020    8.0  2.0        67.0               76.0
 PHO NYL  2020    8.0  2.0        96.0               71.0
 MIN NYL  2020    8.0  5.0        92.0               67.0
 NYL MIN  2020    8.0  5.0        66.0               66.0
 NYL WAS  2020    8.0  7.0        74.0               77.0
 WAS NYL  2020    8.0  7.0        66.0               66.0
 LVA NYL  2020    8.0  9.0        78.0               66.0
 NYL LVA  2020    8.0  9.0        76.0               79.0
 LAS NYL  2020    8.0 11.0        93.0               66.0
 NYL LAS  2020    8.0 11.0        78.0               75.0
 IND NYL  2020

In [83]:
def correct_ytd_opp_max(row):
    opp_games = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]
    return opp_games['team_score'].max() if not opp_games.empty else 0

# Apply clearly
df['ytd_opp_score_max'] = df.apply(correct_ytd_opp_max, axis=1)

# Quick verification clearly
print(
    df.loc[
        (df['year'] == 2020) & ((df['team'] == 'NYL') | (df['opp'] == 'NYL')),
        ['team', 'opp', 'year', 'month', 'day', 'team_score', 'ytd_opp_score_max']
    ].sort_values(['month', 'day']).to_string(index=False)
)

team opp  year  month  day  team_score  ytd_opp_score_max
 NYL SEA  2020    7.0 25.0        71.0                0.0
 SEA NYL  2020    7.0 25.0        87.0                0.0
 DAL NYL  2020    7.0 29.0        93.0               71.0
 NYL DAL  2020    7.0 29.0        80.0               95.0
 ATL NYL  2020    7.0 31.0        84.0               80.0
 NYL ATL  2020    7.0 31.0        78.0              105.0
 NYL PHO  2020    8.0  2.0        67.0              102.0
 PHO NYL  2020    8.0  2.0        96.0               80.0
 MIN NYL  2020    8.0  5.0        92.0               80.0
 NYL MIN  2020    8.0  5.0        66.0               83.0
 NYL WAS  2020    8.0  7.0        74.0              101.0
 WAS NYL  2020    8.0  7.0        66.0               80.0
 LVA NYL  2020    8.0  9.0        78.0               80.0
 NYL LVA  2020    8.0  9.0        76.0              100.0
 LAS NYL  2020    8.0 11.0        93.0               80.0
 NYL LAS  2020    8.0 11.0        78.0               99.0
 IND NYL  2020

In [84]:
# Compute YTD mean opp_score (excluding current game)
df['ytd_team_allowed_mean'] = (
    df.groupby(['team', 'year'])['opp_score']
      .transform(lambda x: x.shift().expanding().mean())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'opp_score', 'ytd_team_allowed_mean']].head(15))

   team  year  opp_score  ytd_team_allowed_mean
0   ATL  2020       95.0               0.000000
1   ATL  2020      100.0              95.000000
2   ATL  2020       78.0              97.500000
3   ATL  2020       93.0              91.000000
4   ATL  2020       81.0              91.500000
5   ATL  2020       93.0              89.400000
6   ATL  2020       85.0              90.000000
7   ATL  2020       93.0              89.285714
8   ATL  2020      100.0              89.750000
9   ATL  2020       96.0              90.888889
10  ATL  2020       92.0              91.400000
11  ATL  2020       98.0              91.454545
12  ATL  2020       93.0              92.000000
13  ATL  2020       75.0              92.076923
14  ATL  2020       88.0              90.857143


In [85]:
# Compute YTD median opp_score (excluding current game)
df['ytd_team_allowed_median'] = (
    df.groupby(['team', 'year'])['opp_score']
      .transform(lambda x: x.shift().expanding().median())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'opp_score', 'ytd_team_allowed_median']].head(15))

   team  year  opp_score  ytd_team_allowed_median
0   ATL  2020       95.0                      0.0
1   ATL  2020      100.0                     95.0
2   ATL  2020       78.0                     97.5
3   ATL  2020       93.0                     95.0
4   ATL  2020       81.0                     94.0
5   ATL  2020       93.0                     93.0
6   ATL  2020       85.0                     93.0
7   ATL  2020       93.0                     93.0
8   ATL  2020      100.0                     93.0
9   ATL  2020       96.0                     93.0
10  ATL  2020       92.0                     93.0
11  ATL  2020       98.0                     93.0
12  ATL  2020       93.0                     93.0
13  ATL  2020       75.0                     93.0
14  ATL  2020       88.0                     93.0


In [86]:
# Compute YTD min opp_score (excluding current game)
df['ytd_team_allowed_min'] = (
    df.groupby(['team', 'year'])['opp_score']
      .transform(lambda x: x.shift().expanding().min())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'opp_score', 'ytd_team_allowed_min']].head(15))

   team  year  opp_score  ytd_team_allowed_min
0   ATL  2020       95.0                   0.0
1   ATL  2020      100.0                  95.0
2   ATL  2020       78.0                  95.0
3   ATL  2020       93.0                  78.0
4   ATL  2020       81.0                  78.0
5   ATL  2020       93.0                  78.0
6   ATL  2020       85.0                  78.0
7   ATL  2020       93.0                  78.0
8   ATL  2020      100.0                  78.0
9   ATL  2020       96.0                  78.0
10  ATL  2020       92.0                  78.0
11  ATL  2020       98.0                  78.0
12  ATL  2020       93.0                  78.0
13  ATL  2020       75.0                  78.0
14  ATL  2020       88.0                  75.0


In [87]:
# Compute YTD max opp_score (excluding current game)
df['ytd_team_allowed_max'] = (
    df.groupby(['team', 'year'])['opp_score']
      .transform(lambda x: x.shift().expanding().max())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'opp_score', 'ytd_team_allowed_max']].head(15))

   team  year  opp_score  ytd_team_allowed_max
0   ATL  2020       95.0                   0.0
1   ATL  2020      100.0                  95.0
2   ATL  2020       78.0                 100.0
3   ATL  2020       93.0                 100.0
4   ATL  2020       81.0                 100.0
5   ATL  2020       93.0                 100.0
6   ATL  2020       85.0                 100.0
7   ATL  2020       93.0                 100.0
8   ATL  2020      100.0                 100.0
9   ATL  2020       96.0                 100.0
10  ATL  2020       92.0                 100.0
11  ATL  2020       98.0                 100.0
12  ATL  2020       93.0                 100.0
13  ATL  2020       75.0                 100.0
14  ATL  2020       88.0                 100.0


In [89]:
# Define the function clearly
def correct_ytd_opp_allowed_mean(row):
    opp_allowed = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]['opp_score']
    return opp_allowed.mean() if not opp_allowed.empty else 0

# Apply safely to df
df['ytd_opp_allowed_mean'] = df.apply(correct_ytd_opp_allowed_mean, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day', 'ytd_opp_allowed_mean']].head(15))

   team  opp  year  month   day  ytd_opp_allowed_mean
0   ATL  DAL  2020    7.0  26.0              0.000000
1   ATL  LVA  2020    7.0  29.0             88.000000
2   ATL  NYL  2020    7.0  31.0             90.000000
3   ATL  IND  2020    8.0   2.0             92.333333
4   ATL  PHO  2020    8.0   4.0             91.750000
5   ATL  SEA  2020    8.0   6.0             75.000000
6   ATL  DAL  2020    8.0   8.0             85.000000
7   ATL  CON  2020    8.0  10.0             83.571429
8   ATL  SEA  2020    8.0  12.0             75.750000
9   ATL  PHO  2020    8.0  14.0             85.444444
10  ATL  CHI  2020    8.0  16.0             83.800000
11  ATL  WAS  2020    8.0  19.0             80.900000
12  ATL  LAS  2020    8.0  21.0             78.454545
13  ATL  MIN  2020    8.0  23.0             76.250000
14  ATL  MIN  2020    8.0  28.0             76.384615


In [90]:
def correct_ytd_opp_allowed_median(row):
    opp_allowed = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]['opp_score']
    return opp_allowed.median() if not opp_allowed.empty else 0

# Apply clearly
df['ytd_opp_allowed_median'] = df.apply(correct_ytd_opp_allowed_median, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day', 'ytd_opp_allowed_median']].head(15))

   team  opp  year  month   day  ytd_opp_allowed_median
0   ATL  DAL  2020    7.0  26.0                     0.0
1   ATL  LVA  2020    7.0  29.0                    88.0
2   ATL  NYL  2020    7.0  31.0                    90.0
3   ATL  IND  2020    8.0   2.0                   100.0
4   ATL  PHO  2020    8.0   4.0                    97.0
5   ATL  SEA  2020    8.0   6.0                    74.0
6   ATL  DAL  2020    8.0   8.0                    81.0
7   ATL  CON  2020    8.0  10.0                    81.0
8   ATL  SEA  2020    8.0  12.0                    72.5
9   ATL  PHO  2020    8.0  14.0                    86.0
10  ATL  CHI  2020    8.0  16.0                    84.5
11  ATL  WAS  2020    8.0  19.0                    82.0
12  ATL  LAS  2020    8.0  21.0                    76.0
13  ATL  MIN  2020    8.0  23.0                    80.0
14  ATL  MIN  2020    8.0  28.0                    80.0


In [91]:
def correct_ytd_opp_allowed_min(row):
    opp_allowed = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]['opp_score']
    return opp_allowed.min() if not opp_allowed.empty else 0

# Apply clearly
df['ytd_opp_allowed_min'] = df.apply(correct_ytd_opp_allowed_min, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day', 'ytd_opp_allowed_min']].head(15))

   team  opp  year  month   day  ytd_opp_allowed_min
0   ATL  DAL  2020    7.0  26.0                  0.0
1   ATL  LVA  2020    7.0  29.0                 88.0
2   ATL  NYL  2020    7.0  31.0                 87.0
3   ATL  IND  2020    8.0   2.0                 76.0
4   ATL  PHO  2020    8.0   4.0                 67.0
5   ATL  SEA  2020    8.0   6.0                 66.0
6   ATL  DAL  2020    8.0   8.0                 73.0
7   ATL  CON  2020    8.0  10.0                 68.0
8   ATL  SEA  2020    8.0  12.0                 66.0
9   ATL  PHO  2020    8.0  14.0                 67.0
10  ATL  CHI  2020    8.0  16.0                 71.0
11  ATL  WAS  2020    8.0  19.0                 68.0
12  ATL  LAS  2020    8.0  21.0                 64.0
13  ATL  MIN  2020    8.0  23.0                 48.0
14  ATL  MIN  2020    8.0  28.0                 48.0


In [92]:
def correct_ytd_opp_allowed_max(row):
    opp_allowed = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]['opp_score']
    return opp_allowed.max() if not opp_allowed.empty else 0

# Apply clearly
df['ytd_opp_allowed_max'] = df.apply(correct_ytd_opp_allowed_max, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day', 'ytd_opp_allowed_max']].head(15))

   team  opp  year  month   day  ytd_opp_allowed_max
0   ATL  DAL  2020    7.0  26.0                  0.0
1   ATL  LVA  2020    7.0  29.0                 88.0
2   ATL  NYL  2020    7.0  31.0                 93.0
3   ATL  IND  2020    8.0   2.0                101.0
4   ATL  PHO  2020    8.0   4.0                106.0
5   ATL  SEA  2020    8.0   6.0                 89.0
6   ATL  DAL  2020    8.0   8.0                105.0
7   ATL  CON  2020    8.0  10.0                100.0
8   ATL  SEA  2020    8.0  12.0                 92.0
9   ATL  PHO  2020    8.0  14.0                106.0
10  ATL  CHI  2020    8.0  16.0                 96.0
11  ATL  WAS  2020    8.0  19.0                 91.0
12  ATL  LAS  2020    8.0  21.0                 96.0
13  ATL  MIN  2020    8.0  23.0                 97.0
14  ATL  MIN  2020    8.0  28.0                 97.0


In [93]:
for col in df.columns:
  print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midran

In [117]:
# Counting stats clearly defined (excluding pct and advanced columns)
features = [
    'team_fg', 'team_fga', 'team_3p', 'team_3pa', 'team_ft', 'team_fta',
    'team_orb', 'team_trb', 'team_ast', 'team_stl', 'team_blk', 'team_tov', 'team_pf',
    'opponent_fg', 'opponent_fga', 'opponent_3p', 'opponent_3pa',
    'opponent_ft', 'opponent_fta', 'opponent_orb', 'opponent_trb',
    'opponent_ast', 'opponent_stl', 'opponent_blk', 'opponent_tov', 'opponent_pf'
]

# Create ytd columns per game
for col in features:
    df[f'ytd_team_{col}_per_game'] = (
        df.groupby(['team', 'year'])[col]
          .transform(lambda x: x.shift().expanding().mean())
          .fillna(0)
    )

# Quick verification of a few columns
print(df[['team', 'year', 'team_fg', 'ytd_team_team_fg_per_game',
          'team_ast', 'ytd_team_team_ast_per_game']].head(15))

   team  year  team_fg  ytd_team_team_fg_per_game  team_ast  \
0   ATL  2020       34                   0.000000        19   
1   ATL  2020       28                  34.000000        12   
2   ATL  2020       28                  31.000000        11   
3   ATL  2020       33                  30.000000        18   
4   ATL  2020       29                  30.750000        15   
5   ATL  2020       31                  30.400000        12   
6   ATL  2020       29                  30.500000        13   
7   ATL  2020       30                  30.285714        21   
8   ATL  2020       27                  30.250000        18   
9   ATL  2020       32                  29.888889        22   
10  ATL  2020       27                  30.100000        11   
11  ATL  2020       38                  29.818182        26   
12  ATL  2020       33                  30.500000        27   
13  ATL  2020       32                  30.692308        23   
14  ATL  2020       31                  30.785714      

In [119]:
# Fix redundant 'team_' in column names
df.columns = [col.replace('ytd_team_team_', 'ytd_team_').replace('ytd_team_opponent_', 'ytd_opponent_') for col in df.columns]

In [125]:
# All team-based percentage columns calculated from cumulative YTD totals
df['ytd_team_fg_pct'] = (df['ytd_team_fg_per_game'] / df['ytd_team_fga_per_game']).fillna(0)
df['ytd_team_3p_pct'] = (df['ytd_team_3p_per_game'] / df['ytd_team_3pa_per_game']).fillna(0)
df['ytd_team_ft_pct'] = (df['ytd_team_ft_per_game'] / df['ytd_team_fta_per_game']).fillna(0)

# Advanced calculations
df['ytd_advanced_trb_pct'] = (
    df['ytd_team_trb_per_game'] /
    (df['ytd_team_trb_per_game'] + df['ytd_opponent_trb_per_game'])
).fillna(0)

df['ytd_advanced_ast_pct'] = (
    df['ytd_team_ast_per_game'] / df['ytd_team_fg_per_game']
).fillna(0)

df['ytd_advanced_blk_pct'] = (
    df['ytd_team_blk_per_game'] / (df['ytd_opponent_fga_per_game'] - df['ytd_opponent_3pa_per_game'])
).fillna(0)

df['ytd_advanced_ftr'] = (
    df['ytd_team_fta_per_game'] / df['ytd_team_fga_per_game']
).fillna(0)

df['ytd_advanced_3par'] = (
    df['ytd_team_3pa_per_game'] / df['ytd_team_fga_per_game']
).fillna(0)

# TS% calculation
df['ytd_team_pts_per_game'] = (
    (df['ytd_team_fg_per_game'] - df['ytd_team_3p_per_game']) * 2 +
    df['ytd_team_3p_per_game'] * 3 +
    df['ytd_team_ft_per_game']
)

df['ytd_advanced_ts_pct'] = (
    df['ytd_team_pts_per_game'] / (2 * (df['ytd_team_fga_per_game'] + 0.44 * df['ytd_team_fta_per_game']))
).fillna(0)

# Quick verification of some columns
print(df[['team', 'year', 'ytd_team_fg_pct', 'ytd_team_3p_pct', 'ytd_team_ft_pct',
          'ytd_advanced_trb_pct', 'ytd_advanced_ast_pct', 'ytd_advanced_ts_pct']].head(15))

   team  year  ytd_team_fg_pct  ytd_team_3p_pct  ytd_team_ft_pct  \
0   ATL  2020         0.000000         0.000000         0.000000   
1   ATL  2020         0.548387         0.411765         0.882353   
2   ATL  2020         0.469697         0.266667         0.812500   
3   ATL  2020         0.434783         0.272727         0.810127   
4   ATL  2020         0.445652         0.287879         0.816092   
5   ATL  2020         0.449704         0.291139         0.830000   
6   ATL  2020         0.450739         0.311321         0.858333   
7   ATL  2020         0.444444         0.328000         0.842105   
8   ATL  2020         0.446494         0.363636         0.825503   
9   ATL  2020         0.436688         0.348837         0.826667   
10  ATL  2020         0.434971         0.354167         0.795181   
11  ATL  2020         0.431579         0.339806         0.777174   
12  ATL  2020         0.441496         0.348214         0.765306   
13  ATL  2020         0.444321         0.353909 

<ipython-input-125-e26c3be54e0d>:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_advanced_ftr'] = (
<ipython-input-125-e26c3be54e0d>:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_advanced_3par'] = (
<ipython-input-125-e26c3be54e0d>:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

In [126]:
for col in df.columns:
  print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opponent_fg
opponent_fga
opponent_fg_pct
opponent_3p
opponent_3pa
opponent_3p_pct
opponent_ft
opponent_fta
opponent_ft_pct
opponent_orb
opponent_trb
opponent_ast
opponent_stl
opponent_blk
opponent_tov
opponent_pf
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midran

In [129]:
# Offensive Four Factors
df['ytd_offensive_efg_pct'] = (
    (df['ytd_team_fg_per_game'] + 0.5 * df['ytd_team_3p_per_game']) / df['ytd_team_fga_per_game']
).fillna(0)

df['ytd_offensive_tov_pct'] = (
    df['ytd_team_tov_per_game'] /
    (df['ytd_team_fga_per_game'] + 0.44 * df['ytd_team_fta_per_game'] + df['ytd_team_tov_per_game'])
).fillna(0)

df['ytd_offensive_orb_pct'] = (
    df['ytd_team_orb_per_game'] /
    (df['ytd_team_orb_per_game'] + df['ytd_opponent_trb_per_game'] - df['ytd_opponent_orb_per_game'])
).fillna(0)

# Defensive Four Factors
df['ytd_defensive_efg_pct'] = (
    (df['ytd_opponent_fg_per_game'] + 0.5 * df['ytd_opponent_3p_per_game']) / df['ytd_opponent_fga_per_game']
).fillna(0)

df['ytd_defensive_tov_pct'] = (
    df['ytd_opponent_tov_per_game'] /
    (df['ytd_opponent_fga_per_game'] + 0.44 * df['ytd_opponent_fta_per_game'] + df['ytd_opponent_tov_per_game'])
).fillna(0)

df['ytd_defensive_drb_pct'] = (
    (df['ytd_team_trb_per_game'] - df['ytd_team_orb_per_game']) /
    (df['ytd_team_trb_per_game'] - df['ytd_team_orb_per_game'] + df['ytd_opponent_orb_per_game'])
).fillna(0)

# Quick verification
print(df[['team', 'year', 'ytd_offensive_efg_pct', 'ytd_offensive_tov_pct', 'ytd_offensive_orb_pct',
          'ytd_defensive_efg_pct', 'ytd_defensive_tov_pct', 'ytd_defensive_drb_pct']].head(15))

   team  year  ytd_offensive_efg_pct  ytd_offensive_tov_pct  \
0   ATL  2020               0.000000               0.000000   
1   ATL  2020               0.604839               0.172117   
2   ATL  2020               0.515152               0.172861   
3   ATL  2020               0.471014               0.162765   
4   ATL  2020               0.480072               0.158058   
5   ATL  2020               0.483728               0.176724   
6   ATL  2020               0.491379               0.177483   
7   ATL  2020               0.487421               0.171688   
8   ATL  2020               0.494465               0.176257   
9   ATL  2020               0.485390               0.174334   
10  ATL  2020               0.484104               0.169374   
11  ATL  2020               0.477632               0.169799   
12  ATL  2020               0.488540               0.168901   
13  ATL  2020               0.492205               0.178104   
14  ATL  2020               0.494301               0.17

<ipython-input-129-6f5175938a62>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_offensive_efg_pct'] = (
<ipython-input-129-6f5175938a62>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_offensive_tov_pct'] = (
<ipython-input-129-6f5175938a62>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.

In [131]:
# List of columns to rename
rename_dict = {
    'ytd_advanced_trb_pct': 'ytd_team_advanced_trb_pct',
    'ytd_advanced_ast_pct': 'ytd_team_advanced_ast_pct',
    'ytd_advanced_blk_pct': 'ytd_team_advanced_blk_pct',
    'ytd_advanced_ftr': 'ytd_team_advanced_ftr',
    'ytd_advanced_3par': 'ytd_team_advanced_3par',
    'ytd_team_pts_per_game': 'ytd_team_pts_per_game',  # this one is already good
    'ytd_advanced_ts_pct': 'ytd_team_advanced_ts_pct',
    'ytd_offensive_efg_pct': 'ytd_team_offensive_efg_pct',
    'ytd_offensive_tov_pct': 'ytd_team_offensive_tov_pct',
    'ytd_offensive_orb_pct': 'ytd_team_offensive_orb_pct',
    'ytd_defensive_efg_pct': 'ytd_team_defensive_efg_pct',
    'ytd_defensive_tov_pct': 'ytd_team_defensive_tov_pct',
    'ytd_defensive_drb_pct': 'ytd_team_defensive_drb_pct'
}

# Rename columns
df.rename(columns=rename_dict, inplace=True)

In [132]:
# Rename 'opponent' to 'opp' for all relevant columns
df.columns = [col.replace('opponent_', 'opp_') for col in df.columns]

# Verify the changes
print([col for col in df.columns if 'opp_' in col])

['opp_score', 'opp_fg', 'opp_fga', 'opp_fg_pct', 'opp_3p', 'opp_3pa', 'opp_3p_pct', 'opp_ft', 'opp_fta', 'opp_ft_pct', 'opp_orb', 'opp_trb', 'opp_ast', 'opp_stl', 'opp_blk', 'opp_tov', 'opp_pf', 'ytd_opp_score_mean', 'ytd_opp_score_median', 'ytd_opp_score_min', 'ytd_opp_score_max', 'ytd_opp_allowed_mean', 'ytd_opp_allowed_median', 'ytd_opp_allowed_min', 'ytd_opp_allowed_max', 'ytd_opp_fg_per_game', 'ytd_opp_fga_per_game', 'ytd_opp_3p_per_game', 'ytd_opp_3pa_per_game', 'ytd_opp_ft_per_game', 'ytd_opp_fta_per_game', 'ytd_opp_orb_per_game', 'ytd_opp_trb_per_game', 'ytd_opp_ast_per_game', 'ytd_opp_stl_per_game', 'ytd_opp_blk_per_game', 'ytd_opp_tov_per_game', 'ytd_opp_pf_per_game']


In [134]:
cols_to_create = [
    'ytd_team_fg_pct', 'ytd_team_3p_pct', 'ytd_team_ft_pct',
    'ytd_team_advanced_trb_pct', 'ytd_team_advanced_ast_pct', 'ytd_team_advanced_blk_pct',
    'ytd_team_advanced_ftr', 'ytd_team_advanced_3par', 'ytd_team_pts_per_game',
    'ytd_team_advanced_ts_pct', 'ytd_team_offensive_efg_pct',
    'ytd_team_offensive_tov_pct', 'ytd_team_offensive_orb_pct',
    'ytd_team_defensive_efg_pct', 'ytd_team_defensive_tov_pct',
    'ytd_team_defensive_drb_pct'
]

for col in cols_to_create:
    new_col = col.replace('ytd_team_', 'ytd_opp_')

    def get_ytd_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(get_ytd_opp, axis=1)

# Quick verification
print(df[['team', 'opp', 'year'] + [col.replace('ytd_team_', 'ytd_opp_') for col in cols_to_create]].head())

<ipython-input-134-8bf2c6464102>:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)
<ipython-input-134-8bf2c6464102>:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)
<ipython-input-134-8bf2c6464102>:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fra

  team  opp  year  ytd_opp_fg_pct  ytd_opp_3p_pct  ytd_opp_ft_pct  \
0  ATL  DAL  2020        0.000000        0.000000        0.000000   
1  ATL  LVA  2020        0.000000        0.000000        0.000000   
2  ATL  NYL  2020        0.348485        0.214286        0.863636   
3  ATL  IND  2020        0.453901        0.369565        0.880952   
4  ATL  PHO  2020        0.505102        0.414286        0.698630   

   ytd_opp_advanced_trb_pct  ytd_opp_advanced_ast_pct  \
0                  0.000000                  0.000000   
1                  0.000000                  0.000000   
2                  0.480519                  0.608696   
3                  0.488889                  0.453125   
4                  0.478947                  0.717172   

   ytd_opp_advanced_blk_pct  ytd_opp_advanced_ftr  ytd_opp_advanced_3par  \
0                  0.000000              0.000000               0.000000   
1                  0.000000              0.000000               0.000000   
2             

<ipython-input-134-8bf2c6464102>:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)


In [137]:
# YTD Advanced ORtg, DRtg, Pace (means of existing game values)
for col in ['advanced_ortg', 'advanced_drtg', 'advanced_pace']:
    df[f'ytd_team_{col}'] = (
        df.groupby(['team', 'year'])[col]
        .transform(lambda x: x.shift().expanding().mean())
        .fillna(0)
    )

# Quick verification
print(df[['team', 'year', 'advanced_ortg', 'ytd_team_advanced_ortg',
          'advanced_drtg', 'ytd_team_advanced_drtg',
          'advanced_pace', 'ytd_team_advanced_pace']].head(10))

  team  year  advanced_ortg  ytd_team_advanced_ortg  advanced_drtg  \
0  ATL  2020          119.4                0.000000          108.0   
1  ATL  2020           83.0              119.400000          118.6   
2  ATL  2020          102.0              101.200000           94.7   
3  ATL  2020           98.4              101.466667          118.9   
4  ATL  2020           89.5              100.700000           98.0   
5  ATL  2020          108.4               98.460000          109.6   
6  ATL  2020          100.7              100.116667          114.1   
7  ATL  2020          102.4              100.200000          116.1   
8  ATL  2020           82.4              100.475000          130.7   
9  ATL  2020          100.9               98.466667          121.1   

   ytd_team_advanced_drtg  advanced_pace  ytd_team_advanced_pace  
0                0.000000           87.9                0.000000  
1              108.000000           84.3               87.900000  
2              113.300000   

<ipython-input-137-fab0841f5222>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'ytd_team_{col}'] = (
<ipython-input-137-fab0841f5222>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'ytd_team_{col}'] = (
<ipython-input-137-fab0841f5222>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[

In [139]:
cols_to_create = ['ytd_team_advanced_ortg', 'ytd_team_advanced_drtg', 'ytd_team_advanced_pace']

for col in cols_to_create:
    new_col = col.replace('ytd_team_', 'ytd_opp_')

    def get_ytd_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(get_ytd_opp, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [col.replace('ytd_team_', 'ytd_opp_') for col in cols_to_create]].head(10))

<ipython-input-139-fe360876d4d2>:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)
<ipython-input-139-fe360876d4d2>:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)


  team  opp  year  month   day  ytd_opp_advanced_ortg  ytd_opp_advanced_drtg  \
0  ATL  DAL  2020    7.0  26.0               0.000000               0.000000   
1  ATL  LVA  2020    7.0  29.0               0.000000               0.000000   
2  ATL  NYL  2020    7.0  31.0              81.400000              99.800000   
3  ATL  IND  2020    8.0   2.0             111.050000             122.550000   
4  ATL  PHO  2020    8.0   4.0             110.833333             119.600000   
5  ATL  SEA  2020    8.0   6.0             104.850000              95.900000   
6  ATL  DAL  2020    8.0   8.0             102.740000             104.060000   
7  ATL  CON  2020    8.0  10.0              96.333333              99.966667   
8  ATL  SEA  2020    8.0  12.0             105.385714              96.528571   
9  ATL  PHO  2020    8.0  14.0             106.887500             102.362500   

   ytd_opp_advanced_pace  
0               0.000000  
1               0.000000  
2              87.200000  
3          

<ipython-input-139-fe360876d4d2>:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)


In [142]:
# List of allowed stats to create from opp columns
allowed_stats = [
    ('opp_fg', 'ytd_team_allowed_fg_per_game'),
    ('opp_fga', 'ytd_team_allowed_fga_per_game'),
    ('opp_3p', 'ytd_team_allowed_3p_per_game'),
    ('opp_3pa', 'ytd_team_allowed_3pa_per_game'),
    ('opp_ft', 'ytd_team_allowed_ft_per_game'),
    ('opp_fta', 'ytd_team_allowed_fta_per_game'),
    ('opp_orb', 'ytd_team_allowed_orb_per_game'),
    ('opp_trb', 'ytd_team_allowed_trb_per_game'),
    ('opp_ast', 'ytd_team_allowed_ast_per_game'),
    ('opp_stl', 'ytd_team_allowed_stl_per_game'),
    ('opp_blk', 'ytd_team_allowed_blk_per_game'),
    ('opp_tov', 'ytd_team_allowed_tov_per_game'),
    ('opp_pf', 'ytd_team_allowed_pf_per_game'),
    ('opp_fg_pct', 'ytd_team_allowed_fg_pct'),
    ('opp_3p_pct', 'ytd_team_allowed_3p_pct'),
    ('opp_ft_pct', 'ytd_team_allowed_ft_pct'),
    ('advanced_ts_pct', 'ytd_team_allowed_advanced_ts_pct'),
    ('advanced_ast_pct', 'ytd_team_allowed_advanced_ast_pct'),
    ('advanced_blk_pct', 'ytd_team_allowed_advanced_blk_pct'),
    ('advanced_ftr', 'ytd_team_allowed_advanced_ftr'),
    ('advanced_3par', 'ytd_team_allowed_advanced_3par'),
    ('offensive_four_factors_efg_pct', 'ytd_team_allowed_offensive_efg_pct'),
    ('offensive_four_factors_tov_pct', 'ytd_team_allowed_offensive_tov_pct'),
    ('offensive_four_factors_orb_pct', 'ytd_team_allowed_offensive_orb_pct'),
    ('defensive_four_factors_efg_pct', 'ytd_team_allowed_defensive_efg_pct'),
    ('defensive_four_factors_tov_pct', 'ytd_team_allowed_defensive_tov_pct'),
    ('defensive_four_factors_drb_pct', 'ytd_team_allowed_defensive_drb_pct')
]

# Compute YTD allowed stats per team
for original_col, new_col in allowed_stats:
    df[new_col] = (
        df.groupby(['team', 'year'])[original_col]
        .transform(lambda x: x.shift().expanding().mean())
        .fillna(0)
    )

# Verify quickly
print(df[['team', 'year'] + [new for _, new in allowed_stats]].head(10))

  team  year  ytd_team_allowed_fg_per_game  ytd_team_allowed_fga_per_game  \
0  ATL  2020                      0.000000                       0.000000   
1  ATL  2020                     34.000000                      78.000000   
2  ATL  2020                     36.000000                      74.500000   
3  ATL  2020                     33.333333                      73.666667   
4  ATL  2020                     33.500000                      71.750000   
5  ATL  2020                     32.600000                      72.400000   
6  ATL  2020                     32.333333                      72.500000   
7  ATL  2020                     32.142857                      71.714286   
8  ATL  2020                     32.625000                      72.375000   
9  ATL  2020                     32.888889                      71.222222   

   ytd_team_allowed_3p_per_game  ytd_team_allowed_3pa_per_game  \
0                      0.000000                       0.000000   
1                   

<ipython-input-142-aa92a140df24>:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipython-input-142-aa92a140df24>:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipython-input-142-aa92a140df24>:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipy

In [143]:
cols_to_create = [
    'ytd_team_allowed_fg_per_game', 'ytd_team_allowed_fga_per_game',
    'ytd_team_allowed_3p_per_game', 'ytd_team_allowed_3pa_per_game',
    'ytd_team_allowed_ft_per_game', 'ytd_team_allowed_fta_per_game',
    'ytd_team_allowed_orb_per_game', 'ytd_team_allowed_trb_per_game',
    'ytd_team_allowed_ast_per_game', 'ytd_team_allowed_stl_per_game',
    'ytd_team_allowed_blk_per_game', 'ytd_team_allowed_tov_per_game',
    'ytd_team_allowed_pf_per_game', 'ytd_team_allowed_fg_pct',
    'ytd_team_allowed_3p_pct', 'ytd_team_allowed_ft_pct',
    'ytd_team_allowed_advanced_ts_pct', 'ytd_team_allowed_advanced_ast_pct',
    'ytd_team_allowed_advanced_blk_pct', 'ytd_team_allowed_advanced_ftr',
    'ytd_team_allowed_advanced_3par', 'ytd_team_allowed_offensive_efg_pct',
    'ytd_team_allowed_offensive_tov_pct', 'ytd_team_allowed_offensive_orb_pct',
    'ytd_team_allowed_defensive_efg_pct', 'ytd_team_allowed_defensive_tov_pct',
    'ytd_team_allowed_defensive_drb_pct'
]

for col in cols_to_create:
    new_col = col.replace('ytd_team_', 'ytd_opp_')

    def get_ytd_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(get_ytd_opp, axis=1)

# Verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [col.replace('ytd_team_', 'ytd_opp_') for col in cols_to_create]].head(10))

<ipython-input-143-8618f64d6216>:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)
<ipython-input-143-8618f64d6216>:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)
<ipython-input-143-8618f64d6216>:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fra

  team  opp  year  month   day  ytd_opp_allowed_fg_per_game  \
0  ATL  DAL  2020    7.0  26.0                     0.000000   
1  ATL  LVA  2020    7.0  29.0                     0.000000   
2  ATL  NYL  2020    7.0  31.0                    34.000000   
3  ATL  IND  2020    8.0   2.0                    36.000000   
4  ATL  PHO  2020    8.0   4.0                    35.333333   
5  ATL  SEA  2020    8.0   6.0                    26.500000   
6  ATL  DAL  2020    8.0   8.0                    30.000000   
7  ATL  CON  2020    8.0  10.0                    30.000000   
8  ATL  SEA  2020    8.0  12.0                    27.285714   
9  ATL  PHO  2020    8.0  14.0                    29.750000   

   ytd_opp_allowed_fga_per_game  ytd_opp_allowed_3p_per_game  \
0                      0.000000                     0.000000   
1                      0.000000                     0.000000   
2                     77.000000                     7.000000   
3                     68.500000                   

<ipython-input-143-8618f64d6216>:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_ytd_opp, axis=1)


In [145]:
# Basic shooting percentages
df['ytd_team_allowed_fg_pct'] = (
    df['ytd_team_allowed_fg_per_game'] / df['ytd_team_allowed_fga_per_game']
).fillna(0)

df['ytd_team_allowed_3p_pct'] = (
    df['ytd_team_allowed_3p_per_game'] / df['ytd_team_allowed_3pa_per_game']
).fillna(0)

df['ytd_team_allowed_ft_pct'] = (
    df['ytd_team_allowed_ft_per_game'] / df['ytd_team_allowed_fta_per_game']
).fillna(0)

df['ytd_opp_allowed_fg_pct'] = (
    df['ytd_opp_allowed_fg_per_game'] / df['ytd_opp_allowed_fga_per_game']
).fillna(0)

df['ytd_opp_allowed_3p_pct'] = (
    df['ytd_opp_allowed_3p_per_game'] / df['ytd_opp_allowed_3pa_per_game']
).fillna(0)

df['ytd_opp_allowed_ft_pct'] = (
    df['ytd_opp_allowed_ft_per_game'] / df['ytd_opp_allowed_fta_per_game']
).fillna(0)

# Advanced/Four Factors - Team Allowed
df['ytd_team_allowed_offensive_efg_pct'] = (
    (df['ytd_team_allowed_fg_per_game'] + 0.5 * df['ytd_team_allowed_3p_per_game']) / df['ytd_team_allowed_fga_per_game']
).fillna(0)

df['ytd_team_allowed_offensive_tov_pct'] = (
    df['ytd_team_allowed_tov_per_game'] /
    (df['ytd_team_allowed_fga_per_game'] + 0.44 * df['ytd_team_allowed_fta_per_game'] + df['ytd_team_allowed_tov_per_game'])
).fillna(0)

df['ytd_team_allowed_offensive_orb_pct'] = (
    df['ytd_team_allowed_orb_per_game'] /
    (df['ytd_team_allowed_orb_per_game'] + df['ytd_opp_allowed_trb_per_game'] - df['ytd_opp_allowed_orb_per_game'])
).fillna(0)

df['ytd_team_allowed_defensive_efg_pct'] = (
    (df['ytd_opp_allowed_fg_per_game'] + 0.5 * df['ytd_opp_allowed_3p_per_game']) / df['ytd_opp_allowed_fga_per_game']
).fillna(0)

df['ytd_team_allowed_defensive_tov_pct'] = (
    df['ytd_opp_allowed_tov_per_game'] /
    (df['ytd_opp_allowed_fga_per_game'] + 0.44 * df['ytd_opp_allowed_fta_per_game'] + df['ytd_opp_allowed_tov_per_game'])
).fillna(0)

df['ytd_team_allowed_defensive_drb_pct'] = (
    (df['ytd_team_allowed_trb_per_game'] - df['ytd_team_allowed_orb_per_game']) /
    (df['ytd_team_allowed_trb_per_game'] - df['ytd_team_allowed_orb_per_game'] + df['ytd_opp_allowed_orb_per_game'])
).fillna(0)

# Advanced/Four Factors - Opponent Allowed
df['ytd_opp_allowed_offensive_efg_pct'] = (
    (df['ytd_opp_allowed_fg_per_game'] + 0.5 * df['ytd_opp_allowed_3p_per_game']) / df['ytd_opp_allowed_fga_per_game']
).fillna(0)

df['ytd_opp_allowed_offensive_tov_pct'] = (
    df['ytd_opp_allowed_tov_per_game'] /
    (df['ytd_opp_allowed_fga_per_game'] + 0.44 * df['ytd_opp_allowed_fta_per_game'] + df['ytd_opp_allowed_tov_per_game'])
).fillna(0)

df['ytd_opp_allowed_offensive_orb_pct'] = (
    df['ytd_opp_allowed_orb_per_game'] /
    (df['ytd_opp_allowed_orb_per_game'] + df['ytd_team_allowed_trb_per_game'] - df['ytd_team_allowed_orb_per_game'])
).fillna(0)

df['ytd_opp_allowed_defensive_efg_pct'] = (
    (df['ytd_team_allowed_fg_per_game'] + 0.5 * df['ytd_team_allowed_3p_per_game']) / df['ytd_team_allowed_fga_per_game']
).fillna(0)

df['ytd_opp_allowed_defensive_tov_pct'] = (
    df['ytd_team_allowed_tov_per_game'] /
    (df['ytd_team_allowed_fga_per_game'] + 0.44 * df['ytd_team_allowed_fta_per_game'] + df['ytd_team_allowed_tov_per_game'])
).fillna(0)

df['ytd_opp_allowed_defensive_drb_pct'] = (
    (df['ytd_opp_allowed_trb_per_game'] - df['ytd_opp_allowed_orb_per_game']) /
    (df['ytd_opp_allowed_trb_per_game'] - df['ytd_opp_allowed_orb_per_game'] + df['ytd_team_allowed_orb_per_game'])
).fillna(0)

# Quick check on recalculated columns
print(df[['team', 'year', 'ytd_team_allowed_fg_pct', 'ytd_team_allowed_offensive_efg_pct',
          'ytd_opp_allowed_fg_pct', 'ytd_opp_allowed_defensive_efg_pct']].head(10))

  team  year  ytd_team_allowed_fg_pct  ytd_team_allowed_offensive_efg_pct  \
0  ATL  2020                 0.000000                            0.000000   
1  ATL  2020                 0.435897                            0.519231   
2  ATL  2020                 0.483221                            0.546980   
3  ATL  2020                 0.452489                            0.511312   
4  ATL  2020                 0.466899                            0.526132   
5  ATL  2020                 0.450276                            0.502762   
6  ATL  2020                 0.445977                            0.496552   
7  ATL  2020                 0.448207                            0.501992   
8  ATL  2020                 0.450777                            0.505181   
9  ATL  2020                 0.461778                            0.524961   

   ytd_opp_allowed_fg_pct  ytd_opp_allowed_defensive_efg_pct  
0                0.000000                           0.000000  
1                0.000000 

In [148]:
# Team YTD
df['ytd_team_advanced_stl_pct'] = (
    df.groupby(['team', 'year'])['advanced_stl_pct']
      .transform(lambda x: x.shift().expanding().mean())
      .fillna(0)
)

# Opponent YTD (mapping from team's YTD above)
def get_ytd_opp_stl_pct(row):
    opp_data = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]
    return opp_data['ytd_team_advanced_stl_pct'].iloc[-1] if not opp_data.empty else 0

df['ytd_opp_advanced_stl_pct'] = df.apply(get_ytd_opp_stl_pct, axis=1)

# Corrected Team Allowed (using opponent's advanced_stl_pct)
df['ytd_team_allowed_advanced_stl_pct'] = (
    df.groupby(['team', 'year'])['advanced_stl_pct']
      .transform(lambda x: x.shift().expanding().mean())
      .fillna(0)
)

# Corrected Opponent Allowed (mapping team's allowed correctly)
def get_ytd_opp_allowed_stl_pct(row):
    opp_data = df[
        (df['team'] == row['opp']) &
        (
            (df['year'] < row['year']) |
            ((df['year'] == row['year']) & (df['month'] < row['month'])) |
            ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
        )
    ]
    return opp_data['ytd_team_allowed_advanced_stl_pct'].iloc[-1] if not opp_data.empty else 0

df['ytd_opp_allowed_advanced_stl_pct'] = df.apply(get_ytd_opp_allowed_stl_pct, axis=1)

# Verify carefully now:
print(df[['team', 'year', 'advanced_stl_pct',
          'ytd_team_advanced_stl_pct',
          'ytd_team_allowed_advanced_stl_pct',
          'ytd_opp_allowed_advanced_stl_pct']].head(10))

  team  year  advanced_stl_pct  ytd_team_advanced_stl_pct  \
0  ATL  2020               8.0                   0.000000   
1  ATL  2020              14.2                   8.000000   
2  ATL  2020               9.7                  11.100000   
3  ATL  2020               7.7                  10.633333   
4  ATL  2020               8.5                   9.900000   
5  ATL  2020               8.3                   9.620000   
6  ATL  2020               5.4                   9.400000   
7  ATL  2020               3.7                   8.828571   
8  ATL  2020               7.8                   8.187500   
9  ATL  2020               6.3                   8.144444   

   ytd_team_allowed_advanced_stl_pct  ytd_opp_allowed_advanced_stl_pct  
0                           0.000000                          0.000000  
1                           8.000000                          0.000000  
2                          11.100000                          5.700000  
3                          10.633333

In [149]:
cols_to_drop = [
    'ytd_team_allowed_fg_per_game',
    'ytd_team_allowed_fga_per_game',
    'ytd_team_allowed_3p_per_game',
    'ytd_team_allowed_3pa_per_game',
    'ytd_team_allowed_ft_per_game',
    'ytd_team_allowed_fta_per_game',
    'ytd_team_allowed_orb_per_game',
    'ytd_team_allowed_trb_per_game',
    'ytd_team_allowed_ast_per_game',
    'ytd_team_allowed_stl_per_game',
    'ytd_team_allowed_blk_per_game',
    'ytd_team_allowed_tov_per_game',
    'ytd_team_allowed_pf_per_game',
    'ytd_team_allowed_fg_pct',
    'ytd_team_allowed_3p_pct',
    'ytd_team_allowed_ft_pct',
    'ytd_team_allowed_advanced_ts_pct',
    'ytd_team_allowed_advanced_ast_pct',
    'ytd_team_allowed_advanced_blk_pct',
    'ytd_team_allowed_advanced_ftr',
    'ytd_team_allowed_advanced_3par',
    'ytd_team_allowed_offensive_efg_pct',
    'ytd_team_allowed_offensive_tov_pct',
    'ytd_team_allowed_offensive_orb_pct',
    'ytd_team_allowed_defensive_efg_pct',
    'ytd_team_allowed_defensive_tov_pct',
    'ytd_team_allowed_defensive_drb_pct',
    'ytd_opp_allowed_fg_per_game',
    'ytd_opp_allowed_fga_per_game',
    'ytd_opp_allowed_3p_per_game',
    'ytd_opp_allowed_3pa_per_game',
    'ytd_opp_allowed_ft_per_game',
    'ytd_opp_allowed_fta_per_game',
    'ytd_opp_allowed_orb_per_game',
    'ytd_opp_allowed_trb_per_game',
    'ytd_opp_allowed_ast_per_game',
    'ytd_opp_allowed_stl_per_game',
    'ytd_opp_allowed_blk_per_game',
    'ytd_opp_allowed_tov_per_game',
    'ytd_opp_allowed_pf_per_game',
    'ytd_opp_allowed_fg_pct',
    'ytd_opp_allowed_3p_pct',
    'ytd_opp_allowed_ft_pct',
    'ytd_opp_allowed_advanced_ts_pct',
    'ytd_opp_allowed_advanced_ast_pct',
    'ytd_opp_allowed_advanced_blk_pct',
    'ytd_opp_allowed_advanced_ftr',
    'ytd_opp_allowed_advanced_3par',
    'ytd_opp_allowed_offensive_efg_pct',
    'ytd_opp_allowed_offensive_tov_pct',
    'ytd_opp_allowed_offensive_orb_pct',
    'ytd_opp_allowed_defensive_efg_pct',
    'ytd_opp_allowed_defensive_tov_pct',
    'ytd_opp_allowed_defensive_drb_pct'
]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')


In [152]:
df.drop(columns=['ytd_team_allowed_advanced_stl_pct', 'ytd_opp_allowed_advanced_stl_pct'], inplace=True, errors='ignore')

# Confirm the drop
print([col for col in df.columns if '_allowed_' in col])

['ytd_team_allowed_mean', 'ytd_team_allowed_median', 'ytd_team_allowed_min', 'ytd_team_allowed_max', 'ytd_opp_allowed_mean', 'ytd_opp_allowed_median', 'ytd_opp_allowed_min', 'ytd_opp_allowed_max']


In [155]:
# Create ytd_team_allowed_fg_per_game
df['ytd_team_allowed_fg_per_game'] = (
    df.groupby(['team', 'year'])['opp_fg']
      .transform(lambda x: x.shift().expanding().mean())
      .fillna(0)
)

# Verify carefully (first 10 rows for one team)
print(
    df[df['team'] == 'ATL'][['team', 'year', 'opp', 'opp_fg', 'ytd_team_allowed_fg_per_game']].head(10)
)

  team  year  opp  opp_fg  ytd_team_allowed_fg_per_game
0  ATL  2020  DAL      34                      0.000000
1  ATL  2020  LVA      38                     34.000000
2  ATL  2020  NYL      28                     36.000000
3  ATL  2020  IND      34                     33.333333
4  ATL  2020  PHO      29                     33.500000
5  ATL  2020  SEA      31                     32.600000
6  ATL  2020  DAL      31                     32.333333
7  ATL  2020  CON      36                     32.142857
8  ATL  2020  SEA      35                     32.625000
9  ATL  2020  PHO      32                     32.888889


<ipython-input-155-f035bb47e46a>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_team_allowed_fg_per_game'] = (


In [156]:
# counting stats including advanced_stl_pct
counting_cols = [
    'opp_fga', 'opp_3p', 'opp_3pa', 'opp_ft', 'opp_fta',
    'opp_orb', 'opp_trb', 'opp_ast', 'opp_stl', 'opp_blk',
    'opp_tov', 'opp_pf', 'advanced_stl_pct'
]

# loop and create allowed columns
for col in counting_cols:
    df[f'ytd_team_allowed_{col}_per_game'] = (
        df.groupby(['team', 'year'])[col]
          .transform(lambda x: x.shift().expanding().mean())
          .fillna(0)
    )

# quick verification
print(df[df['team'] == 'ATL'][['team', 'year', 'opp', 'opp_fga', 'ytd_team_allowed_opp_fga_per_game', 'opp_ast', 'ytd_team_allowed_opp_ast_per_game', 'advanced_stl_pct', 'ytd_team_allowed_advanced_stl_pct_per_game']].head(10))

<ipython-input-156-ae951d87fa77>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'ytd_team_allowed_{col}_per_game'] = (
<ipython-input-156-ae951d87fa77>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'ytd_team_allowed_{col}_per_game'] = (
<ipython-input-156-ae951d87fa77>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

  team  year  opp  opp_fga  ytd_team_allowed_opp_fga_per_game  opp_ast  \
0  ATL  2020  DAL       78                           0.000000       18   
1  ATL  2020  LVA       71                          78.000000       16   
2  ATL  2020  NYL       72                          74.500000       13   
3  ATL  2020  IND       66                          73.666667       26   
4  ATL  2020  PHO       75                          71.750000       19   
5  ATL  2020  SEA       73                          72.400000       21   
6  ATL  2020  DAL       67                          72.500000       17   
7  ATL  2020  CON       77                          71.714286       19   
8  ATL  2020  SEA       62                          72.375000       31   
9  ATL  2020  PHO       65                          71.222222       24   

   ytd_team_allowed_opp_ast_per_game  advanced_stl_pct  \
0                           0.000000               8.0   
1                          18.000000              14.2   
2          

<ipython-input-156-ae951d87fa77>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'ytd_team_allowed_{col}_per_game'] = (
<ipython-input-156-ae951d87fa77>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'ytd_team_allowed_{col}_per_game'] = (
<ipython-input-156-ae951d87fa77>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

In [157]:
# Calculated allowed percentages
df['ytd_team_allowed_fg_pct'] = (
    df['ytd_team_allowed_fg_per_game'] / df['ytd_team_allowed_opp_fga_per_game']
).fillna(0)

df['ytd_team_allowed_3p_pct'] = (
    df['ytd_team_allowed_opp_3p_per_game'] / df['ytd_team_allowed_opp_3pa_per_game']
).fillna(0)

df['ytd_team_allowed_ft_pct'] = (
    df['ytd_team_allowed_opp_ft_per_game'] / df['ytd_team_allowed_opp_fta_per_game']
).fillna(0)

df['ytd_team_allowed_advanced_ts_pct'] = (
    df['ytd_team_allowed_fg_per_game'] /
    (2 * (df['ytd_team_allowed_opp_fga_per_game'] + 0.44 * df['ytd_team_allowed_opp_fta_per_game']))
).fillna(0)

df['ytd_team_allowed_offensive_efg_pct'] = (
    (df['ytd_team_allowed_fg_per_game'] + 0.5 * df['ytd_team_allowed_opp_3p_per_game']) /
    df['ytd_team_allowed_opp_fga_per_game']
).fillna(0)

df['ytd_team_allowed_offensive_tov_pct'] = (
    df['ytd_team_allowed_opp_tov_per_game'] /
    (df['ytd_team_allowed_opp_fga_per_game'] + 0.44 * df['ytd_team_allowed_opp_fta_per_game'] + df['ytd_team_allowed_opp_tov_per_game'])
).fillna(0)

df['ytd_team_allowed_offensive_orb_pct'] = (
    df['ytd_team_allowed_opp_orb_per_game'] /
    (df['ytd_team_allowed_opp_orb_per_game'] + df['ytd_team_trb_per_game'] - df['ytd_team_orb_per_game'])
).fillna(0)

df['ytd_team_allowed_defensive_efg_pct'] = (
    (df['ytd_team_fg_per_game'] + 0.5 * df['ytd_team_3p_per_game']) / df['ytd_team_fga_per_game']
).fillna(0)

df['ytd_team_allowed_defensive_tov_pct'] = (
    df['ytd_team_tov_per_game'] /
    (df['ytd_team_fga_per_game'] + 0.44 * df['ytd_team_fta_per_game'] + df['ytd_team_tov_per_game'])
).fillna(0)

df['ytd_team_allowed_defensive_drb_pct'] = (
    (df['ytd_team_allowed_opp_trb_per_game'] - df['ytd_team_allowed_opp_orb_per_game']) /
    (df['ytd_team_allowed_opp_trb_per_game'] - df['ytd_team_allowed_opp_orb_per_game'] + df['ytd_team_orb_per_game'])
).fillna(0)

# Quick verification
print(df[df['team'] == 'ATL'][[
    'team', 'year', 'ytd_team_allowed_fg_pct', 'ytd_team_allowed_3p_pct',
    'ytd_team_allowed_ft_pct', 'ytd_team_allowed_offensive_efg_pct',
    'ytd_team_allowed_offensive_tov_pct', 'ytd_team_allowed_offensive_orb_pct'
]].head(10))

  team  year  ytd_team_allowed_fg_pct  ytd_team_allowed_3p_pct  \
0  ATL  2020                 0.000000                 0.000000   
1  ATL  2020                 0.435897                 0.464286   
2  ATL  2020                 0.483221                 0.513514   
3  ATL  2020                 0.452489                 0.393939   
4  ATL  2020                 0.466899                 0.395349   
5  ATL  2020                 0.450276                 0.358491   
6  ATL  2020                 0.445977                 0.346457   
7  ATL  2020                 0.448207                 0.355263   
8  ATL  2020                 0.450777                 0.357955   
9  ATL  2020                 0.461778                 0.393204   

   ytd_team_allowed_ft_pct  ytd_team_allowed_offensive_efg_pct  \
0                 0.000000                            0.000000   
1                 0.700000                            0.519231   
2                 0.695652                            0.546980   
3        

<ipython-input-157-301d53cf1e08>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_team_allowed_fg_pct'] = (
<ipython-input-157-301d53cf1e08>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ytd_team_allowed_3p_pct'] = (
<ipython-input-157-301d53cf1e08>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fr

In [158]:
# Quick verification with actual columns included
print(df[df['team'] == 'ATL'][[
    'team', 'year',
    'opp_fg_pct', 'ytd_team_allowed_fg_pct',
    'opp_3p_pct', 'ytd_team_allowed_3p_pct',
    'opp_ft_pct', 'ytd_team_allowed_ft_pct',
    'offensive_four_factors_efg_pct', 'ytd_team_allowed_offensive_efg_pct',
    'offensive_four_factors_tov_pct', 'ytd_team_allowed_offensive_tov_pct',
    'offensive_four_factors_orb_pct', 'ytd_team_allowed_offensive_orb_pct'
]].head(10))

  team  year  opp_fg_pct  ytd_team_allowed_fg_pct  opp_3p_pct  \
0  ATL  2020       0.436                 0.000000       0.464   
1  ATL  2020       0.535                 0.435897       0.667   
2  ATL  2020       0.389                 0.483221       0.241   
3  ATL  2020       0.515                 0.452489       0.400   
4  ATL  2020       0.387                 0.466899       0.200   
5  ATL  2020       0.425                 0.450276       0.286   
6  ATL  2020       0.463                 0.445977       0.400   
7  ATL  2020       0.468                 0.448207       0.375   
8  ATL  2020       0.565                 0.450777       0.600   
9  ATL  2020       0.492                 0.461778       0.480   

   ytd_team_allowed_3p_pct  opp_ft_pct  ytd_team_allowed_ft_pct  \
0                 0.000000       0.700                 0.000000   
1                 0.464286       0.692                 0.700000   
2                 0.513514       0.882                 0.695652   
3               

In [159]:
# Loop over ALL ytd_team_allowed_ columns to create ytd_opp_allowed_
allowed_cols = [col for col in df.columns if col.startswith('ytd_team_allowed_')]

for col in allowed_cols:
    new_col = col.replace('ytd_team_allowed_', 'ytd_opp_allowed_')

    def map_team_to_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(map_team_to_opp, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [c.replace('ytd_team_allowed_', 'ytd_opp_allowed_') for c in allowed_cols]].head(15))

<ipython-input-159-3abda3510e5a>:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(map_team_to_opp, axis=1)
<ipython-input-159-3abda3510e5a>:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(map_team_to_opp, axis=1)
<ipython-input-159-3abda3510e5a>:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragme

   team  opp  year  month   day  ytd_opp_allowed_mean  ytd_opp_allowed_median  \
0   ATL  DAL  2020    7.0  26.0              0.000000                     0.0   
1   ATL  LVA  2020    7.0  29.0              0.000000                     0.0   
2   ATL  NYL  2020    7.0  31.0             87.000000                    87.0   
3   ATL  IND  2020    8.0   2.0            100.500000                   100.5   
4   ATL  PHO  2020    8.0   4.0            100.000000                    99.0   
5   ATL  SEA  2020    8.0   6.0             75.250000                    73.0   
6   ATL  DAL  2020    8.0   8.0             83.800000                    80.0   
7   ATL  CON  2020    8.0  10.0             80.833333                    79.5   
8   ATL  SEA  2020    8.0  12.0             76.428571                    74.0   
9   ATL  PHO  2020    8.0  14.0             85.000000                    82.5   
10  ATL  CHI  2020    8.0  16.0             84.555556                    86.0   
11  ATL  WAS  2020    8.0  1

<ipython-input-159-3abda3510e5a>:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(map_team_to_opp, axis=1)


In [162]:
# 10-game rolling average for team_score
df['rolling_10_team_score_mean'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'team_score', 'rolling_10_team_score_mean']].head(30))

    team  year  team_score  rolling_10_team_score_mean
0    ATL  2020       105.0                    0.000000
1    ATL  2020        70.0                  105.000000
2    ATL  2020        84.0                   87.500000
3    ATL  2020        77.0                   86.333333
4    ATL  2020        74.0                   84.000000
5    ATL  2020        92.0                   82.000000
6    ATL  2020        75.0                   83.666667
7    ATL  2020        82.0                   82.428571
8    ATL  2020        63.0                   82.375000
9    ATL  2020        80.0                   80.222222
10   ATL  2020        67.0                   80.200000
11   ATL  2020        91.0                   76.400000
12   ATL  2020        85.0                   78.500000
13   ATL  2020        78.0                   78.600000
14   ATL  2020        79.0                   78.700000
15   ATL  2020        79.0                   79.200000
16   ATL  2020       102.0                   77.900000
17   ATL  

In [167]:
# Calculate rolling 10-game median for team_score (excluding current game)
df['rolling_10_team_score_median'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().rolling(10, min_periods=1).median())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'team_score', 'rolling_10_team_score_median']].head(15))

   team  year  team_score  rolling_10_team_score_median
0   ATL  2020       105.0                           0.0
1   ATL  2020        70.0                         105.0
2   ATL  2020        84.0                          87.5
3   ATL  2020        77.0                          84.0
4   ATL  2020        74.0                          80.5
5   ATL  2020        92.0                          77.0
6   ATL  2020        75.0                          80.5
7   ATL  2020        82.0                          77.0
8   ATL  2020        63.0                          79.5
9   ATL  2020        80.0                          77.0
10  ATL  2020        67.0                          78.5
11  ATL  2020        91.0                          76.0
12  ATL  2020        85.0                          78.5
13  ATL  2020        78.0                          78.5
14  ATL  2020        79.0                          79.0


<ipython-input-167-fe67c48dd3ef>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_score_median'] = (


In [168]:
# Calculate rolling 10-game minimum for team_score (excluding current game)
df['rolling_10_team_score_min'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().rolling(10, min_periods=1).min())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'team_score', 'rolling_10_team_score_min']].head(15))

   team  year  team_score  rolling_10_team_score_min
0   ATL  2020       105.0                        0.0
1   ATL  2020        70.0                      105.0
2   ATL  2020        84.0                       70.0
3   ATL  2020        77.0                       70.0
4   ATL  2020        74.0                       70.0
5   ATL  2020        92.0                       70.0
6   ATL  2020        75.0                       70.0
7   ATL  2020        82.0                       70.0
8   ATL  2020        63.0                       70.0
9   ATL  2020        80.0                       63.0
10  ATL  2020        67.0                       63.0
11  ATL  2020        91.0                       63.0
12  ATL  2020        85.0                       63.0
13  ATL  2020        78.0                       63.0
14  ATL  2020        79.0                       63.0


<ipython-input-168-e3072310eb89>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_score_min'] = (


In [169]:
# Calculate rolling 10-game maximum for team_score (excluding current game)
df['rolling_10_team_score_max'] = (
    df.groupby(['team', 'year'])['team_score']
      .transform(lambda x: x.shift().rolling(10, min_periods=1).max())
      .fillna(0)
)

# Quick verification
print(df[['team', 'year', 'team_score', 'rolling_10_team_score_max']].head(15))

   team  year  team_score  rolling_10_team_score_max
0   ATL  2020       105.0                        0.0
1   ATL  2020        70.0                      105.0
2   ATL  2020        84.0                      105.0
3   ATL  2020        77.0                      105.0
4   ATL  2020        74.0                      105.0
5   ATL  2020        92.0                      105.0
6   ATL  2020        75.0                      105.0
7   ATL  2020        82.0                      105.0
8   ATL  2020        63.0                      105.0
9   ATL  2020        80.0                      105.0
10  ATL  2020        67.0                      105.0
11  ATL  2020        91.0                       92.0
12  ATL  2020        85.0                       92.0
13  ATL  2020        78.0                       92.0
14  ATL  2020        79.0                       92.0


<ipython-input-169-adb1154c31bb>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_score_max'] = (


In [171]:
# List of columns to map from team to opp
team_cols = [
    'rolling_10_team_score_mean',
    'rolling_10_team_score_median',
    'rolling_10_team_score_min',
    'rolling_10_team_score_max'
]

for col in team_cols:
    opp_col = col.replace('team', 'opp')

    # Map rolling team stats to opponent clearly (excluding current game)
    def get_rolling_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[opp_col] = df.apply(get_rolling_opp, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [c.replace('team', 'opp') for c in team_cols]].head(15))

<ipython-input-171-cfc8b62088f4>:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[opp_col] = df.apply(get_rolling_opp, axis=1)
<ipython-input-171-cfc8b62088f4>:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[opp_col] = df.apply(get_rolling_opp, axis=1)
<ipython-input-171-cfc8b62088f4>:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragme

   team  opp  year  month   day  rolling_10_opp_score_mean  \
0   ATL  DAL  2020    7.0  26.0                   0.000000   
1   ATL  LVA  2020    7.0  29.0                   0.000000   
2   ATL  NYL  2020    7.0  31.0                  71.000000   
3   ATL  IND  2020    8.0   2.0                  91.000000   
4   ATL  PHO  2020    8.0   4.0                  92.666667   
5   ATL  SEA  2020    8.0   6.0                  82.250000   
6   ATL  DAL  2020    8.0   8.0                  82.600000   
7   ATL  CON  2020    8.0  10.0                  78.000000   
8   ATL  SEA  2020    8.0  12.0                  83.285714   
9   ATL  PHO  2020    8.0  14.0                  88.750000   
10  ATL  CHI  2020    8.0  16.0                  86.777778   
11  ATL  WAS  2020    8.0  19.0                  78.777778   
12  ATL  LAS  2020    8.0  21.0                  86.200000   
13  ATL  MIN  2020    8.0  23.0                  81.700000   
14  ATL  MIN  2020    8.0  28.0                  84.100000   

    rol

<ipython-input-171-cfc8b62088f4>:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[opp_col] = df.apply(get_rolling_opp, axis=1)


In [173]:
# Corrected counting columns (percentages removed)
counting_cols = [
    'team_fg', 'team_fga', 'team_3p', 'team_3pa',
    'team_ft', 'team_fta', 'team_orb', 'team_trb',
    'team_ast', 'team_stl', 'team_blk', 'team_tov',
    'team_pf', 'advanced_stl_pct'
]

# Loop for team counting cols, rolling 10-game averages
for col in counting_cols:
    df[f'rolling_10_{col}'] = (
        df.groupby(['team', 'year'])[col]
          .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
          .fillna(0)
    )

# Quick verification
print(df[['team', 'year'] + [f'rolling_10_{col}' for col in counting_cols]].head(15))

<ipython-input-173-03cffa1132a1>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'rolling_10_{col}'] = (
<ipython-input-173-03cffa1132a1>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'rolling_10_{col}'] = (
<ipython-input-173-03cffa1132a1>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

   team  year  rolling_10_team_fg  rolling_10_team_fga  rolling_10_team_3p  \
0   ATL  2020            0.000000             0.000000            0.000000   
1   ATL  2020           34.000000            62.000000            7.000000   
2   ATL  2020           31.000000            66.000000            6.000000   
3   ATL  2020           30.000000            69.000000            5.000000   
4   ATL  2020           30.750000            69.000000            4.750000   
5   ATL  2020           30.400000            67.600000            4.600000   
6   ATL  2020           30.500000            67.666667            5.500000   
7   ATL  2020           30.285714            68.142857            5.857143   
8   ATL  2020           30.250000            67.750000            6.500000   
9   ATL  2020           29.888889            68.444444            6.666667   
10  ATL  2020           30.100000            69.200000            6.800000   
11  ATL  2020           29.400000            69.800000          

<ipython-input-173-03cffa1132a1>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'rolling_10_{col}'] = (
<ipython-input-173-03cffa1132a1>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'rolling_10_{col}'] = (
<ipython-input-173-03cffa1132a1>:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()

In [175]:
df.rename(columns={'rolling_10_advanced_stl_pct': 'rolling_10_team_advanced_stl_pct'}, inplace=True)

In [177]:
# Only correct counting columns to create
counting_cols = [
    ('advanced_ortg', 'rolling_10_team_advanced_ortg'),
    ('advanced_drtg', 'rolling_10_team_advanced_drtg'),
    ('advanced_pace', 'rolling_10_team_advanced_pace')
]

for original_col, new_col in counting_cols:
    df[new_col] = (
        df.groupby(['team', 'year'])[original_col]
          .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
          .fillna(0)
    )

# Quick verification
print(df[['team', 'year', 'advanced_ortg', 'rolling_10_team_advanced_ortg',
          'advanced_drtg', 'rolling_10_team_advanced_drtg',
          'advanced_pace', 'rolling_10_team_advanced_pace']].head(15))

   team  year  advanced_ortg  rolling_10_team_advanced_ortg  advanced_drtg  \
0   ATL  2020          119.4                       0.000000          108.0   
1   ATL  2020           83.0                     119.400000          118.6   
2   ATL  2020          102.0                     101.200000           94.7   
3   ATL  2020           98.4                     101.466667          118.9   
4   ATL  2020           89.5                     100.700000           98.0   
5   ATL  2020          108.4                      98.460000          109.6   
6   ATL  2020          100.7                     100.116667          114.1   
7   ATL  2020          102.4                     100.200000          116.1   
8   ATL  2020           82.4                     100.475000          130.7   
9   ATL  2020          100.9                      98.466667          121.1   
10  ATL  2020           82.7                      98.710000          113.6   
11  ATL  2020          119.4                      95.040000     

<ipython-input-177-fd28489a7d10>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipython-input-177-fd28489a7d10>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipython-input-177-fd28489a7d10>:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (


In [180]:
cols_to_create = [
    'rolling_10_team_fg',
    'rolling_10_team_fga',
    'rolling_10_team_3p',
    'rolling_10_team_3pa',
    'rolling_10_team_ft',
    'rolling_10_team_fta',
    'rolling_10_team_orb',
    'rolling_10_team_trb',
    'rolling_10_team_ast',
    'rolling_10_team_stl',
    'rolling_10_team_blk',
    'rolling_10_team_tov',
    'rolling_10_team_pf',
    'rolling_10_team_advanced_stl_pct',
    'rolling_10_team_advanced_ortg',
    'rolling_10_team_advanced_drtg',
    'rolling_10_team_advanced_pace',
    'rolling_10_team_fg_pct',
    'rolling_10_team_3p_pct',
    'rolling_10_team_ft_pct',
    'rolling_10_team_offensive_efg_pct',
    'rolling_10_team_offensive_tov_pct'
]

for col in cols_to_create:
    opp_col = col.replace('rolling_10_team_', 'rolling_10_opp_')

    def get_rolling_10_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[opp_col] = df.apply(get_rolling_10_opp, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [col.replace('rolling_10_team_', 'rolling_10_opp_') for col in cols_to_create]].head(10))

<ipython-input-180-b5f2753d9571>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[opp_col] = df.apply(get_rolling_10_opp, axis=1)
<ipython-input-180-b5f2753d9571>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[opp_col] = df.apply(get_rolling_10_opp, axis=1)
<ipython-input-180-b5f2753d9571>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-

  team  opp  year  month   day  rolling_10_opp_fg  rolling_10_opp_fga  \
0  ATL  DAL  2020    7.0  26.0           0.000000            0.000000   
1  ATL  LVA  2020    7.0  29.0           0.000000            0.000000   
2  ATL  NYL  2020    7.0  31.0          23.000000           66.000000   
3  ATL  IND  2020    8.0   2.0          32.000000           70.500000   
4  ATL  PHO  2020    8.0   4.0          33.000000           65.333333   
5  ATL  SEA  2020    8.0   6.0          31.500000           67.750000   
6  ATL  DAL  2020    8.0   8.0          30.600000           74.400000   
7  ATL  CON  2020    8.0  10.0          27.833333           67.166667   
8  ATL  SEA  2020    8.0  12.0          30.428571           67.714286   
9  ATL  PHO  2020    8.0  14.0          31.375000           67.250000   

   rolling_10_opp_3p  rolling_10_opp_3pa  rolling_10_opp_ft  ...  \
0           0.000000            0.000000           0.000000  ...   
1           0.000000            0.000000           0.000000 

<ipython-input-180-b5f2753d9571>:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[opp_col] = df.apply(get_rolling_10_opp, axis=1)


In [185]:
# Calculate rolling 10 game percentages and advanced metrics
df['rolling_10_team_fg_pct'] = (
    df['rolling_10_team_fg'] / df['rolling_10_team_fga']
).fillna(0)

df['rolling_10_team_3p_pct'] = (
    df['rolling_10_team_3p'] / df['rolling_10_team_3pa']
).fillna(0)

df['rolling_10_team_ft_pct'] = (
    df['rolling_10_team_ft'] / df['rolling_10_team_fta']
).fillna(0)

df['rolling_10_team_offensive_efg_pct'] = (
    (df['rolling_10_team_fg'] + 0.5 * df['rolling_10_team_3p']) / df['rolling_10_team_fga']
).fillna(0)

df['rolling_10_team_offensive_tov_pct'] = (
    df['rolling_10_team_tov'] /
    (df['rolling_10_team_fga'] + 0.44 * df['rolling_10_team_fta'] + df['rolling_10_team_tov'])
).fillna(0)

df['rolling_10_team_offensive_orb_pct'] = (
    df['rolling_10_team_orb'] /
    (df['rolling_10_team_orb'] + df['rolling_10_opp_trb'] - df['rolling_10_opp_orb'])
).fillna(0)

df['rolling_10_team_defensive_efg_pct'] = (
    (df['rolling_10_opp_fg'] + 0.5 * df['rolling_10_opp_3p']) / df['rolling_10_opp_fga']
).fillna(0)

df['rolling_10_team_defensive_tov_pct'] = (
    df['rolling_10_opp_tov'] /
    (df['rolling_10_opp_fga'] + 0.44 * df['rolling_10_opp_fta'] + df['rolling_10_opp_tov'])
).fillna(0)

df['rolling_10_team_defensive_drb_pct'] = (
    (df['rolling_10_team_trb'] - df['rolling_10_team_orb']) /
    (df['rolling_10_team_trb'] - df['rolling_10_team_orb'] + df['rolling_10_opp_orb'])
).fillna(0)

df['rolling_10_team_advanced_ts_pct'] = (
    df['rolling_10_team_score_mean'] / (2 * (df['rolling_10_team_fga'] + 0.44 * df['rolling_10_team_fta']))
).fillna(0)

df['rolling_10_team_advanced_trb_pct'] = (
    df['rolling_10_team_trb'] / (df['rolling_10_team_trb'] + df['rolling_10_opp_trb'])
).fillna(0)

df['rolling_10_team_advanced_ast_pct'] = (
    df['rolling_10_team_ast'] / df['rolling_10_team_fg']
).fillna(0)

df['rolling_10_team_advanced_blk_pct'] = (
    df['rolling_10_team_blk'] / df['rolling_10_opp_fga']
).fillna(0)

df['rolling_10_team_advanced_ftr'] = (
    df['rolling_10_team_fta'] / df['rolling_10_team_fga']
).fillna(0)

df['rolling_10_team_advanced_3par'] = (
    df['rolling_10_team_3pa'] / df['rolling_10_team_fga']
).fillna(0)

# Quick verification
cols = [
    'team', 'year', 'rolling_10_team_fg_pct', 'rolling_10_team_3p_pct',
    'rolling_10_team_ft_pct', 'rolling_10_team_offensive_efg_pct',
    'rolling_10_team_defensive_efg_pct', 'rolling_10_team_advanced_ts_pct'
]
print(df[cols].head(15))

   team  year  rolling_10_team_fg_pct  rolling_10_team_3p_pct  \
0   ATL  2020                0.000000                0.000000   
1   ATL  2020                0.548387                0.411765   
2   ATL  2020                0.469697                0.266667   
3   ATL  2020                0.434783                0.272727   
4   ATL  2020                0.445652                0.287879   
5   ATL  2020                0.449704                0.291139   
6   ATL  2020                0.450739                0.311321   
7   ATL  2020                0.444444                0.328000   
8   ATL  2020                0.446494                0.363636   
9   ATL  2020                0.436688                0.348837   
10  ATL  2020                0.434971                0.354167   
11  ATL  2020                0.421203                0.333333   
12  ATL  2020                0.436155                0.368715   
13  ATL  2020                0.447178                0.377660   
14  ATL  2020            

<ipython-input-185-dd314c62010e>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_fg_pct'] = (
<ipython-input-185-dd314c62010e>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_3p_pct'] = (
<ipython-input-185-dd314c62010e>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

In [187]:
# List of columns for which we need opp-side rolling features
cols_team_to_opp = [
    'rolling_10_team_fg_pct',
    'rolling_10_team_3p_pct',
    'rolling_10_team_ft_pct',
    'rolling_10_team_offensive_efg_pct',
    'rolling_10_team_offensive_tov_pct',
    'rolling_10_team_offensive_orb_pct',
    'rolling_10_team_defensive_efg_pct',
    'rolling_10_team_defensive_tov_pct',
    'rolling_10_team_defensive_drb_pct',
    'rolling_10_team_advanced_ts_pct',
    'rolling_10_team_advanced_trb_pct',
    'rolling_10_team_advanced_ast_pct',
    'rolling_10_team_advanced_blk_pct',
    'rolling_10_team_advanced_ftr',
    'rolling_10_team_advanced_3par'
]

# Create rolling_10_opp_ versions
for col in cols_team_to_opp:
    new_col = col.replace('rolling_10_team_', 'rolling_10_opp_')

    def get_opp_rolling_10(row):
        # Get opponent previous games data
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        # Return the most recent rolling value or 0 if none
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(get_opp_rolling_10, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [c.replace('rolling_10_team_', 'rolling_10_opp_') for c in cols_team_to_opp]].head(10))

<ipython-input-187-0810c3f34574>:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_opp_rolling_10, axis=1)
<ipython-input-187-0810c3f34574>:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_opp_rolling_10, axis=1)
<ipython-input-187-0810c3f34574>:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-

  team  opp  year  month   day  rolling_10_opp_fg_pct  rolling_10_opp_3p_pct  \
0  ATL  DAL  2020    7.0  26.0               0.000000               0.000000   
1  ATL  LVA  2020    7.0  29.0               0.000000               0.000000   
2  ATL  NYL  2020    7.0  31.0               0.348485               0.214286   
3  ATL  IND  2020    8.0   2.0               0.453901               0.369565   
4  ATL  PHO  2020    8.0   4.0               0.505102               0.414286   
5  ATL  SEA  2020    8.0   6.0               0.464945               0.322917   
6  ATL  DAL  2020    8.0   8.0               0.411290               0.298507   
7  ATL  CON  2020    8.0  10.0               0.414392               0.285714   
8  ATL  SEA  2020    8.0  12.0               0.449367               0.370370   
9  ATL  PHO  2020    8.0  14.0               0.466543               0.341176   

   rolling_10_opp_ft_pct  rolling_10_opp_offensive_efg_pct  \
0               0.000000                          0.00000

<ipython-input-187-0810c3f34574>:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(get_opp_rolling_10, axis=1)


In [189]:
# fix redundant '_opp_' in allowed columns
rename_dict = {
    'ytd_team_allowed_opp_fga_per_game': 'ytd_team_allowed_fga_per_game',
    'ytd_team_allowed_opp_3p_per_game': 'ytd_team_allowed_3p_per_game',
    'ytd_team_allowed_opp_3pa_per_game': 'ytd_team_allowed_3pa_per_game',
    'ytd_team_allowed_opp_ft_per_game': 'ytd_team_allowed_ft_per_game',
    'ytd_team_allowed_opp_fta_per_game': 'ytd_team_allowed_fta_per_game',
    'ytd_team_allowed_opp_orb_per_game': 'ytd_team_allowed_orb_per_game',
    'ytd_team_allowed_opp_trb_per_game': 'ytd_team_allowed_trb_per_game',
    'ytd_team_allowed_opp_ast_per_game': 'ytd_team_allowed_ast_per_game',
    'ytd_team_allowed_opp_stl_per_game': 'ytd_team_allowed_stl_per_game',
    'ytd_team_allowed_opp_blk_per_game': 'ytd_team_allowed_blk_per_game',
    'ytd_team_allowed_opp_tov_per_game': 'ytd_team_allowed_tov_per_game',
    'ytd_team_allowed_opp_pf_per_game': 'ytd_team_allowed_pf_per_game',
    'ytd_opp_allowed_opp_fga_per_game': 'ytd_opp_allowed_fga_per_game',
    'ytd_opp_allowed_opp_3p_per_game': 'ytd_opp_allowed_3p_per_game',
    'ytd_opp_allowed_opp_3pa_per_game': 'ytd_opp_allowed_3pa_per_game',
    'ytd_opp_allowed_opp_ft_per_game': 'ytd_opp_allowed_ft_per_game',
    'ytd_opp_allowed_opp_fta_per_game': 'ytd_opp_allowed_fta_per_game',
    'ytd_opp_allowed_opp_orb_per_game': 'ytd_opp_allowed_orb_per_game',
    'ytd_opp_allowed_opp_trb_per_game': 'ytd_opp_allowed_trb_per_game',
    'ytd_opp_allowed_opp_ast_per_game': 'ytd_opp_allowed_ast_per_game',
    'ytd_opp_allowed_opp_stl_per_game': 'ytd_opp_allowed_stl_per_game',
    'ytd_opp_allowed_opp_blk_per_game': 'ytd_opp_allowed_blk_per_game',
    'ytd_opp_allowed_opp_tov_per_game': 'ytd_opp_allowed_tov_per_game',
    'ytd_opp_allowed_opp_pf_per_game': 'ytd_opp_allowed_pf_per_game'
}

df.rename(columns=rename_dict, inplace=True)

In [191]:
# Counting columns to create rolling-10 allowed
counting_cols = [
    'opp_fg', 'opp_fga', 'opp_3p', 'opp_3pa', 'opp_ft', 'opp_fta',
    'opp_orb', 'opp_trb', 'opp_ast', 'opp_stl', 'opp_blk',
    'opp_tov', 'opp_pf', 'advanced_stl_pct'
]

# Create rolling-10 allowed columns (fixing naming redundancy)
for col in counting_cols:
    clean_col = col.replace('opp_', '') if col.startswith('opp_') else col
    new_col = f'rolling_10_team_allowed_{clean_col}_per_game'
    df[new_col] = (
        df.groupby(['team', 'year'])[col]
          .transform(lambda x: x.shift().rolling(window=10, min_periods=1).mean())
          .fillna(0)
    )

# Quick verification on one team
print(df[df['team'] == 'ATL'][[
    'team', 'year', 'opp_fg', 'rolling_10_team_allowed_fg_per_game',
    'opp_ast', 'rolling_10_team_allowed_ast_per_game',
    'advanced_stl_pct', 'rolling_10_team_allowed_advanced_stl_pct_per_game'
]].head(15))

<ipython-input-191-5304e9ce6bed>:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipython-input-191-5304e9ce6bed>:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipython-input-191-5304e9ce6bed>:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = (
<ipy

   team  year  opp_fg  rolling_10_team_allowed_fg_per_game  opp_ast  \
0   ATL  2020      34                             0.000000       18   
1   ATL  2020      38                            34.000000       16   
2   ATL  2020      28                            36.000000       13   
3   ATL  2020      34                            33.333333       26   
4   ATL  2020      29                            33.500000       19   
5   ATL  2020      31                            32.600000       21   
6   ATL  2020      31                            32.333333       17   
7   ATL  2020      36                            32.142857       19   
8   ATL  2020      35                            32.625000       31   
9   ATL  2020      32                            32.888889       24   
10  ATL  2020      37                            32.800000       25   
11  ATL  2020      35                            33.100000       27   
12  ATL  2020      30                            32.800000       21   
13  AT

In [193]:
# rolling-10 allowed percentage columns
df['rolling_10_team_allowed_fg_pct'] = (
    df['rolling_10_team_allowed_fg_per_game'] / df['rolling_10_team_allowed_fga_per_game']
).fillna(0)

df['rolling_10_team_allowed_3p_pct'] = (
    df['rolling_10_team_allowed_3p_per_game'] / df['rolling_10_team_allowed_3pa_per_game']
).fillna(0)

df['rolling_10_team_allowed_ft_pct'] = (
    df['rolling_10_team_allowed_ft_per_game'] / df['rolling_10_team_allowed_fta_per_game']
).fillna(0)

df['rolling_10_team_allowed_advanced_ts_pct'] = (
    df['rolling_10_team_allowed_fg_per_game'] /
    (2 * (df['rolling_10_team_allowed_fga_per_game'] + 0.44 * df['rolling_10_team_allowed_fta_per_game']))
).fillna(0)

df['rolling_10_team_allowed_offensive_efg_pct'] = (
    (df['rolling_10_team_allowed_fg_per_game'] + 0.5 * df['rolling_10_team_allowed_3p_per_game']) /
    df['rolling_10_team_allowed_fga_per_game']
).fillna(0)

df['rolling_10_team_allowed_offensive_tov_pct'] = (
    df['rolling_10_team_allowed_tov_per_game'] /
    (df['rolling_10_team_allowed_fga_per_game'] + 0.44 * df['rolling_10_team_allowed_fta_per_game'] + df['rolling_10_team_allowed_tov_per_game'])
).fillna(0)

df['rolling_10_team_allowed_offensive_orb_pct'] = (
    df['rolling_10_team_allowed_orb_per_game'] /
    (df['rolling_10_team_allowed_orb_per_game'] + df['rolling_10_team_trb'] - df['rolling_10_team_orb'])
).fillna(0)

df['rolling_10_team_allowed_defensive_efg_pct'] = (
    (df['rolling_10_team_fg'] + 0.5 * df['rolling_10_team_3p']) / df['rolling_10_team_fga']
).fillna(0)

df['rolling_10_team_allowed_defensive_tov_pct'] = (
    df['rolling_10_team_tov'] /
    (df['rolling_10_team_fga'] + 0.44 * df['rolling_10_team_fta'] + df['rolling_10_team_tov'])
).fillna(0)

df['rolling_10_team_allowed_defensive_drb_pct'] = (
    (df['rolling_10_team_allowed_trb_per_game'] - df['rolling_10_team_allowed_orb_per_game']) /
    (df['rolling_10_team_allowed_trb_per_game'] - df['rolling_10_team_allowed_orb_per_game'] + df['rolling_10_team_orb'])
).fillna(0)

# Quick verification
print(df[df['team'] == 'ATL'][[
    'team', 'year',
    'rolling_10_team_allowed_fg_pct', 'rolling_10_team_allowed_3p_pct',
    'rolling_10_team_allowed_ft_pct', 'rolling_10_team_allowed_offensive_efg_pct',
    'rolling_10_team_allowed_offensive_tov_pct', 'rolling_10_team_allowed_offensive_orb_pct'
]].head(15))

   team  year  rolling_10_team_allowed_fg_pct  rolling_10_team_allowed_3p_pct  \
0   ATL  2020                        0.000000                        0.000000   
1   ATL  2020                        0.435897                        0.464286   
2   ATL  2020                        0.483221                        0.513514   
3   ATL  2020                        0.452489                        0.393939   
4   ATL  2020                        0.466899                        0.395349   
5   ATL  2020                        0.450276                        0.358491   
6   ATL  2020                        0.445977                        0.346457   
7   ATL  2020                        0.448207                        0.355263   
8   ATL  2020                        0.450777                        0.357955   
9   ATL  2020                        0.461778                        0.393204   
10  ATL  2020                        0.464589                        0.402597   
11  ATL  2020               

<ipython-input-193-9aebc02c5d93>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_allowed_fg_pct'] = (
<ipython-input-193-9aebc02c5d93>:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rolling_10_team_allowed_3p_pct'] = (
<ipython-input-193-9aebc02c5d93>:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use 

In [195]:
# get list of rolling-10 team allowed columns
allowed_cols = [col for col in df.columns if col.startswith('rolling_10_team_allowed_')]

# loop to create rolling-10 opp allowed columns
for col in allowed_cols:
    new_col = col.replace('rolling_10_team_allowed_', 'rolling_10_opp_allowed_')

    def map_team_to_opp(row):
        opp_data = df[
            (df['team'] == row['opp']) &
            (
                (df['year'] < row['year']) |
                ((df['year'] == row['year']) & (df['month'] < row['month'])) |
                ((df['year'] == row['year']) & (df['month'] == row['month']) & (df['day'] < row['day']))
            )
        ]
        return opp_data[col].iloc[-1] if not opp_data.empty else 0

    df[new_col] = df.apply(map_team_to_opp, axis=1)

# Quick verification
print(df[['team', 'opp', 'year', 'month', 'day'] + [c.replace('rolling_10_team_allowed_', 'rolling_10_opp_allowed_') for c in allowed_cols]].head(15))

<ipython-input-195-81a14a453263>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(map_team_to_opp, axis=1)
<ipython-input-195-81a14a453263>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(map_team_to_opp, axis=1)
<ipython-input-195-81a14a453263>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragme

   team  opp  year  month   day  rolling_10_opp_allowed_fg_per_game  \
0   ATL  DAL  2020    7.0  26.0                            0.000000   
1   ATL  LVA  2020    7.0  29.0                            0.000000   
2   ATL  NYL  2020    7.0  31.0                           34.000000   
3   ATL  IND  2020    8.0   2.0                           36.000000   
4   ATL  PHO  2020    8.0   4.0                           35.333333   
5   ATL  SEA  2020    8.0   6.0                           26.500000   
6   ATL  DAL  2020    8.0   8.0                           30.000000   
7   ATL  CON  2020    8.0  10.0                           30.000000   
8   ATL  SEA  2020    8.0  12.0                           27.285714   
9   ATL  PHO  2020    8.0  14.0                           29.750000   
10  ATL  CHI  2020    8.0  16.0                           31.777778   
11  ATL  WAS  2020    8.0  19.0                           28.444444   
12  ATL  LAS  2020    8.0  21.0                           28.000000   
13  AT

<ipython-input-195-81a14a453263>:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_col] = df.apply(map_team_to_opp, axis=1)


In [198]:
# Dictionary for renaming rolling averages clearly to include '_per_game'
rename_dict = {
    'rolling_10_team_fg': 'rolling_10_team_fg_per_game',
    'rolling_10_team_fga': 'rolling_10_team_fga_per_game',
    'rolling_10_team_3p': 'rolling_10_team_3p_per_game',
    'rolling_10_team_3pa': 'rolling_10_team_3pa_per_game',
    'rolling_10_team_ft': 'rolling_10_team_ft_per_game',
    'rolling_10_team_fta': 'rolling_10_team_fta_per_game',
    'rolling_10_team_orb': 'rolling_10_team_orb_per_game',
    'rolling_10_team_trb': 'rolling_10_team_trb_per_game',
    'rolling_10_team_ast': 'rolling_10_team_ast_per_game',
    'rolling_10_team_stl': 'rolling_10_team_stl_per_game',
    'rolling_10_team_blk': 'rolling_10_team_blk_per_game',
    'rolling_10_team_tov': 'rolling_10_team_tov_per_game',
    'rolling_10_team_pf': 'rolling_10_team_pf_per_game',
    'rolling_10_opp_fg': 'rolling_10_opp_fg_per_game',
    'rolling_10_opp_fga': 'rolling_10_opp_fga_per_game',
    'rolling_10_opp_3p': 'rolling_10_opp_3p_per_game',
    'rolling_10_opp_3pa': 'rolling_10_opp_3pa_per_game',
    'rolling_10_opp_ft': 'rolling_10_opp_ft_per_game',
    'rolling_10_opp_fta': 'rolling_10_opp_fta_per_game',
    'rolling_10_opp_orb': 'rolling_10_opp_orb_per_game',
    'rolling_10_opp_trb': 'rolling_10_opp_trb_per_game',
    'rolling_10_opp_ast': 'rolling_10_opp_ast_per_game',
    'rolling_10_opp_stl': 'rolling_10_opp_stl_per_game',
    'rolling_10_opp_blk': 'rolling_10_opp_blk_per_game',
    'rolling_10_opp_tov': 'rolling_10_opp_tov_per_game',
    'rolling_10_opp_pf': 'rolling_10_opp_pf_per_game',
    'rolling_10_team_advanced_ortg': 'rolling_10_team_advanced_ortg_per_game',
    'rolling_10_team_advanced_drtg': 'rolling_10_team_advanced_drtg_per_game',
    'rolling_10_team_advanced_pace': 'rolling_10_team_advanced_pace_per_game',
    'rolling_10_opp_advanced_ortg': 'rolling_10_opp_advanced_ortg_per_game',
    'rolling_10_opp_advanced_drtg': 'rolling_10_opp_advanced_drtg_per_game',
    'rolling_10_opp_advanced_pace': 'rolling_10_opp_advanced_pace_per_game'
}

# Apply renaming
df.rename(columns=rename_dict, inplace=True)

# Quick verification of renaming
print([col for col in df.columns if 'rolling_10_' in col and '_per_game' in col])

['rolling_10_team_fg_per_game', 'rolling_10_team_fga_per_game', 'rolling_10_team_3p_per_game', 'rolling_10_team_3pa_per_game', 'rolling_10_team_ft_per_game', 'rolling_10_team_fta_per_game', 'rolling_10_team_orb_per_game', 'rolling_10_team_trb_per_game', 'rolling_10_team_ast_per_game', 'rolling_10_team_stl_per_game', 'rolling_10_team_blk_per_game', 'rolling_10_team_tov_per_game', 'rolling_10_team_pf_per_game', 'rolling_10_team_advanced_ortg_per_game', 'rolling_10_team_advanced_drtg_per_game', 'rolling_10_team_advanced_pace_per_game', 'rolling_10_opp_fg_per_game', 'rolling_10_opp_fga_per_game', 'rolling_10_opp_3p_per_game', 'rolling_10_opp_3pa_per_game', 'rolling_10_opp_ft_per_game', 'rolling_10_opp_fta_per_game', 'rolling_10_opp_orb_per_game', 'rolling_10_opp_trb_per_game', 'rolling_10_opp_ast_per_game', 'rolling_10_opp_stl_per_game', 'rolling_10_opp_blk_per_game', 'rolling_10_opp_tov_per_game', 'rolling_10_opp_pf_per_game', 'rolling_10_opp_advanced_ortg_per_game', 'rolling_10_opp_advan

In [200]:
# Step 1: Create unique game IDs for easy matching
df['game_id'] = df['team'] + '_' + df['opp'] + '_' + df['year'].astype(str) + '_' + df['month'].astype(str) + '_' + df['day'].astype(str)
df['opp_game_id'] = df['opp'] + '_' + df['team'] + '_' + df['year'].astype(str) + '_' + df['month'].astype(str) + '_' + df['day'].astype(str)

# Quick verify
print(df[['team', 'opp', 'game_id', 'opp_game_id']].head())

  team  opp                game_id            opp_game_id
0  ATL  DAL  ATL_DAL_2020_7.0_26.0  DAL_ATL_2020_7.0_26.0
1  ATL  LVA  ATL_LVA_2020_7.0_29.0  LVA_ATL_2020_7.0_29.0
2  ATL  NYL  ATL_NYL_2020_7.0_31.0  NYL_ATL_2020_7.0_31.0
3  ATL  IND   ATL_IND_2020_8.0_2.0   IND_ATL_2020_8.0_2.0
4  ATL  PHO   ATL_PHO_2020_8.0_4.0   PHO_ATL_2020_8.0_4.0


<ipython-input-200-2643cc858837>:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['game_id'] = df['team'] + '_' + df['opp'] + '_' + df['year'].astype(str) + '_' + df['month'].astype(str) + '_' + df['day'].astype(str)
<ipython-input-200-2643cc858837>:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['opp_game_id'] = df['opp'] + '_' + df['team'] + '_' + df['year'].astype(str) + '_' + df['month'].astype(str) + '_' + df['day'].astype(str)


In [201]:
# Step 2: Define the persona columns
persona_cols = [
    'all_around_star', 'and_one_machine', 'catch_and_shoot', 'corner_3_specialist',
    'defensive_anchor', 'defensive_rebounder', 'efficient_scorer', 'elite_scorer',
    'fast_break_threat', 'floor_general', 'free_throw_generator', 'glass_cleaner',
    'heave_chucker', 'impact_bench', 'midrange_sniper', 'offensive_hub',
    'offensive_rebounder', 'playmaker', 'plus_minus_driver', 'rim_protector',
    'self_creator', 'slasher', 'steal_artist', 'stretch_big',
    'three_point_specialist', 'turnover_prone', 'volume_shooter'
]

# Create temporary df for opponent personas
opp_personas = df[['game_id'] + persona_cols].copy()

# Rename columns for merging clearly
opp_personas.rename(columns={col: f'opp_{col}' for col in persona_cols}, inplace=True)
opp_personas.rename(columns={'game_id': 'opp_game_id'}, inplace=True)

# Merge to get opp_ columns in original df
df = df.merge(opp_personas, on='opp_game_id', how='left')

# Drop temporary columns if desired (or keep for verification)
df.drop(['game_id', 'opp_game_id'], axis=1, inplace=True)

# Quick verification
verify_cols = ['team', 'opp', 'year', 'month', 'day', 'stretch_big', 'opp_stretch_big']
print(df[verify_cols].head(10))

  team  opp  year  month   day  stretch_big  opp_stretch_big
0  ATL  DAL  2020    7.0  26.0          0.0              0.0
1  ATL  LVA  2020    7.0  29.0          0.0              0.0
2  ATL  NYL  2020    7.0  31.0          0.0              1.0
3  ATL  IND  2020    8.0   2.0          0.0              0.0
4  ATL  PHO  2020    8.0   4.0          0.0              0.0
5  ATL  SEA  2020    8.0   6.0          0.0              0.0
6  ATL  DAL  2020    8.0   8.0          0.0              0.0
7  ATL  CON  2020    8.0  10.0          0.0              0.0
8  ATL  SEA  2020    8.0  12.0          0.0              0.0
9  ATL  PHO  2020    8.0  14.0          0.0              0.0


In [202]:
for col in df.columns:
  print(col)

team
year
home_away
opp
win_loss
team_score
opp_score
team_fg
team_fga
team_fg_pct
team_3p
team_3pa
team_3p_pct
team_ft
team_fta
team_ft_pct
team_orb
team_trb
team_ast
team_stl
team_blk
team_tov
team_pf
opp_fg
opp_fga
opp_fg_pct
opp_3p
opp_3pa
opp_3p_pct
opp_ft
opp_fta
opp_ft_pct
opp_orb
opp_trb
opp_ast
opp_stl
opp_blk
opp_tov
opp_pf
day
month
advanced_ortg
advanced_drtg
advanced_pace
advanced_ftr
advanced_3par
advanced_ts_pct
advanced_trb_pct
advanced_ast_pct
advanced_stl_pct
advanced_blk_pct
offensive_four_factors_efg_pct
offensive_four_factors_tov_pct
offensive_four_factors_orb_pct
defensive_four_factors_efg_pct
defensive_four_factors_tov_pct
defensive_four_factors_drb_pct
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_prot

In [203]:
# Save DataFrame as CSV file locally in Colab
df.to_csv('wnba_final_df.csv', index=False)

# Download directly to your local machine
from google.colab import files
files.download('wnba_final_df.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>